# Native starting-source audit: declared official API namespace repair

Retry11 proved a compatible Python 3.12.13 CPU environment, then failed before
inventory because Evaluator's public module does not reexport extract_snapshot.
Both earlier notebooks, actual failures and environment checkpoints remain.

This declared code retry resolves helpers from the actual official verifier's
implementation globals/module, checks the signatures against the retained
official calls, and preserves their exact source, module identity and hashes.
It creates and exports an audit receipt before imports/API capture. The source
inventory program, problem-only plan and strict byte comparison are unchanged.
No model, repository code, grader tests or reference solution is executed.

Use the same older notebook's original 2026-07-01 environment, Accelerator None,
Internet off, official competition data and metric wheelhouse. The existing
runtime/wheel preflight remains. Preserve all setup and audit checkpoints.
Source equivalence is unknown until an actual completed inventory; a starting
tree match cannot establish a coding repair, deadline or private-grader result.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, base64, zipfile, io, subprocess, importlib, glob
from datetime import datetime, timezone
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_SOURCE_AUDIT = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
from pathlib import Path
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifact_paths = sorted(p for p in run_root.rglob('*') if p.is_file()
                            and not p.is_symlink() and p.name != 'evidence_bundle.zip')
    manifest = []
    for artifact in artifact_paths:
        resolved = artifact.resolve()
        if not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact escaped its directory')
        manifest.append({'path': artifact.relative_to(run_root).as_posix(),
                         'bytes': artifact.stat().st_size,
                         'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for artifact in artifact_paths:
            archive.write(artifact, artifact.relative_to(run_root).as_posix())
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    (run_root / 'evidence_bundle.zip').write_bytes(bundle_bytes)
    report = {'sha256': hashlib.sha256(bundle_bytes).hexdigest(), 'bytes': len(bundle_bytes), 'file_count': len(manifest)}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
ENVIRONMENT_SOURCE_BINDING = json.loads('{"original_notebook_path": "kaggle/10_mobile_official_source_audit.ipynb", "original_notebook_sha256": "d47689e723b18f8ff69b74bc03184d07313a0c2226bfe9b012401168e9f893a1", "prior_failed_run": "reports/kaggle_runs/20261004_mobile_source_audit", "prior_script_version_id": 355107253, "unchanged_payload_cell_sha256": {"4": "9498eedb36f091e566c35205458a08124b1e3131427255d38f19c6bdedf77aec", "5": "21589914180a2029cb7066a0e423c0e6edf41fe7d3215cd15cde302778e1b7bf", "6": "81f56136e2788f60851a6db36d0d5a5fb7f7f66e5ff83fef82b0af026cc63cda"}}')

if IS_KAGGLE and RUN_SOURCE_AUDIT:
    ENVIRONMENT_RUN = Path('/kaggle/working') / ('native_source_audit_environment_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    # This creates a durable setup receipt before any package mutation.
    CPU_ENVIRONMENT_RECEIPT = prepare_source_audit_environment(
        WHEELHOUSE_DIR, ENVIRONMENT_RUN,
        {'swegemma': '0.2.7', 'adk-submission': '0.2.12', 'adk-eval-core': '0.1.0'},
        export_run, ENVIRONMENT_SOURCE_BINDING)
    RAW_SOURCE_AUDIT_EXPORT = export_run
    def export_run(run_root, *, emit=True):
        # Attach the verified setup bytes to each original audit checkpoint.
        import shutil
        destination = Path(run_root) / 'environment_setup'
        destination.mkdir(exist_ok=True)
        for name in ('environment_receipt.json', 'organizer_install_stdout.txt', 'organizer_install_stderr.txt'):
            source = ENVIRONMENT_RUN / name
            if source.is_file():
                shutil.copyfile(source, destination / name)
        return RAW_SOURCE_AUDIT_EXPORT(run_root, emit=emit)
else:
    print('Local definitions only; no CPU environment install or official source audit executed')


In [ ]:
"""Compare actual organizer snapshot bytes with a frozen problem-only mobile input.

This checks starting-state equivalence, not inference, tests or repair success.
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path
import shlex


INVENTORY_PROGRAM = r'''
from pathlib import Path
import hashlib, json, os, stat
root = Path('/workspace')
files, total = {}, 0
for directory, directories, names in os.walk(root, followlinks=False):
    directories[:] = sorted(name for name in directories if name != '.git')
    for name in directories:
        if (Path(directory) / name).is_symlink():
            raise ValueError('Source directory is a symlink')
    for name in sorted(names):
        path = Path(directory) / name
        relative = path.relative_to(root).as_posix()
        info = path.lstat()
        if not stat.S_ISREG(info.st_mode):
            raise ValueError('Source includes a nonregular member: ' + relative)
        total += info.st_size
        if total > 256 * 1024 * 1024 or len(files) >= 20000:
            raise ValueError('Official source exceeds bounded audit capacity')
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            while chunk := stream.read(1048576):
                digest.update(chunk)
        files[relative] = {'sha256': digest.hexdigest(), 'bytes': info.st_size,
                           'mode': stat.S_IMODE(info.st_mode)}
if not files:
    raise ValueError('Official extraction yielded no source files')
print('MOBILE_SOURCE_INVENTORY=' + json.dumps({'files': files, 'source_bytes': total}, sort_keys=True))
'''


def compare_inventories(mobile, official):
    """Exact byte identities; do not relax mismatches into an equivalence claim."""
    def validate(values):
        if not isinstance(values, dict) or not values:
            raise ValueError('An empty or missing inventory cannot establish equivalence')
        for name, value in values.items():
            if (not isinstance(name, str) or not name or '\\' in name or ':' in name
                    or any(part in {'', '.', '..', '.git'} for part in name.split('/'))):
                raise ValueError('Invalid inventory source path')
            if (not isinstance(value, str) or len(value) != 64
                    or any(character not in '0123456789abcdef' for character in value)):
                raise ValueError('Invalid inventory digest')
    validate(mobile)
    validate(official)
    shared = set(mobile) & set(official)
    changed = sorted(name for name in shared if mobile[name] != official[name])
    absent_mobile = sorted(set(official) - set(mobile))
    absent_official = sorted(set(mobile) - set(official))
    return {'full_source_file_bytes_equivalent': mobile == official,
            'matching_files': sum(mobile[name] == official[name] for name in shared),
            'mobile_file_count': len(mobile), 'official_file_count': len(official),
            'different_bytes': changed, 'missing_from_mobile': absent_mobile,
            'missing_from_official': absent_official,
            'file_permissions_equivalence_established': False,
            'excluded_from_both_inventories': ['.git']}


def sha_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def run_mobile_source_audit(data_dir, run_root, plan, checkpoint_exporter):
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    if torch.cuda.device_count() != 0:
        raise ValueError('Select Accelerator None for the source audit')
    if (sha_file(data_dir / 'tasks.jsonl') != plan['tasks_sha256']
            or sha_file(data_dir / 'HARNESS_README.md') != plan['guide_sha256']):
        raise ValueError('Organizer input differs from the frozen contract')
    manifest = plan['mobile_input_manifest']
    problem = plan['problem_only_task']
    if (set(problem) != {'instance_id', 'repo', 'base_commit', 'problem_statement', 'hints_text'}
            or manifest['reference_fields_included'] is not False
            or (manifest['task_id'], manifest['repo'], manifest['base_commit']) !=
               (problem['instance_id'], problem['repo'], problem['base_commit'])):
        raise ValueError('Mobile input identity differs or includes reference fields')
    packages = {name: importlib.metadata.version(name) for name in plan['expected_core_packages']}
    if packages != plan['expected_core_packages']:
        raise ValueError('Installed organizer packages differ from the observed contract')
    task = next(row for row in load_tasks(data_dir / 'tasks.jsonl') if row.instance_id == manifest['task_id'])
    if any(getattr(task, name) != value for name, value in problem.items()):
        raise ValueError('Official problem-only task differs from mobile input')
    module = inspect.getmodule(Evaluator)
    verifier_source = inspect.getsource(module.verify_task)
    if hashlib.sha256(verifier_source.encode()).hexdigest() != plan['expected_verify_task_sha256']:
        raise ValueError('Official verifier differs from the recorded fixture version')
    run_root.mkdir(parents=True, exist_ok=False)
    for name in ('verify_task', 'resolve_task_snapshot_paths', 'extract_snapshot', 'sandbox_start', 'sandbox_exec', 'sandbox_stop'):
        (run_root / (name + '_api.py')).write_text(inspect.getsource(getattr(module, name)), encoding='utf-8')
    (run_root / 'plan.json').write_text(json.dumps(plan, indent=2) + '\n', encoding='utf-8')
    snapshot, base, task_patch = module.resolve_task_snapshot_paths(data_dir / 'snapshots', task.instance_id, task.repo)
    report = {'scope': 'official_starting_source_byte_audit_not_model_or_grading_result',
              'created_utc': datetime.now(timezone.utc).isoformat(), 'task_id': task.instance_id,
              'status': 'started', 'model_inference_executed': False, 'tests_executed': False,
              'reference_solution_text_accessed_by_audit': False, 'task_correctness': None,
              'official_leaderboard_score': None, 'core_packages': packages,
              'mobile_input_manifest_sha256': plan['mobile_input_manifest_sha256'],
              'input_packet_sha256': plan['input_packet_sha256'],
              'official_snapshot_sha256': {str(p.relative_to(data_dir)): sha_file(p) for p in (snapshot, base, task_patch) if p is not None},
              'comparison': None, 'sandbox_cleanup_completed': False}
    def save():
        (run_root / 'source_audit.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    save()
    checkpoint_exporter(run_root)
    config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl', snapshots_dir=data_dir / 'snapshots',
        results_dir=run_root / 'unused_results', submission_dir=run_root / 'unused_agent', models={},
        sandbox='subprocess', timeout_seconds=300, wheels_dir=data_dir / 'wheels',
        graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'))
    evaluator = Evaluator(config)
    async def inventory():
        sandbox_id = await module.sandbox_start(evaluator.docker)
        try:
            await asyncio.to_thread(module.extract_snapshot, evaluator.docker, sandbox_id, snapshot,
                                    patch_path=task_patch, base_snapshot_path=base)
            result = await module.sandbox_exec(evaluator.docker, sandbox_id,
                'cd /workspace && python -c ' + shlex.quote(INVENTORY_PROGRAM))
            (run_root / 'inventory_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'inventory_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['inventory_exit_code'] = result.exit_code
            if result.exit_code != 0:
                raise RuntimeError('Official snapshot inventory failed; inspect retained stdout/stderr')
            records = [line[len('MOBILE_SOURCE_INVENTORY='):] for line in (result.stdout or '').splitlines()
                       if line.startswith('MOBILE_SOURCE_INVENTORY=')]
            if len(records) != 1:
                raise ValueError('Missing or ambiguous official source inventory')
            actual = json.loads(records[0])
            (run_root / 'official_source_inventory.json').write_text(json.dumps(actual, indent=2) + '\n', encoding='utf-8')
            report['official_inventory_sha256'] = sha_file(run_root / 'official_source_inventory.json')
            report['comparison'] = compare_inventories(manifest['workspace_sha256'],
                                                       {name: row['sha256'] for name, row in actual['files'].items()})
        finally:
            await module.sandbox_stop(evaluator.docker, sandbox_id)
            report['sandbox_cleanup_completed'] = True
            save()
    try:
        with ThreadPoolExecutor(max_workers=1) as pool:
            pool.submit(lambda: asyncio.run(inventory())).result()
        report['status'] = 'completed'
        save()
        checkpoint_exporter(run_root)
        return report
    except Exception as error:
        report.update(status='audit_exception', error_type=type(error).__name__, error=str(error))
        save()
        checkpoint_exporter(run_root)
        raise


In [ ]:
"""Declared source-audit code retry using the verifier's actual implementation.

The inventory and comparison are unchanged. Namespace/API failures are captured
inside a started audit receipt; reference solutions and repository code never run.
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path
import shlex
import traceback



def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


def source_audit_runtime():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    return torch, EvalConfig, Evaluator, load_tasks


def run_mobile_source_audit(data_dir, run_root, plan, checkpoint_exporter):
    run_root = Path(run_root)
    run_root.mkdir(parents=True, exist_ok=False)
    manifest = plan["mobile_input_manifest"]
    problem = plan["problem_only_task"]
    report = {
        "scope": "official_starting_source_byte_audit_not_model_or_grading_result",
        "audit_code_revision": "verifier_implementation_namespace_v2",
        "created_utc": datetime.now(timezone.utc).isoformat(), "task_id": manifest["task_id"],
        "status": "started", "model_inference_executed": False, "tests_executed": False,
        "reference_solution_text_accessed_by_audit": False, "task_correctness": None,
        "official_leaderboard_score": None, "core_packages": None,
        "mobile_input_manifest_sha256": plan["mobile_input_manifest_sha256"],
        "input_packet_sha256": plan["input_packet_sha256"],
        "inventory_program_sha256": hashlib.sha256(INVENTORY_PROGRAM.encode()).hexdigest(),
        "official_snapshot_sha256": None, "official_api_provenance": {},
        "comparison": None, "sandbox_cleanup_completed": False,
    }

    def save():
        (run_root / "source_audit.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")

    try:
        # This is deliberately before imports, input validation or API capture.
        (run_root / "plan.json").write_text(json.dumps(plan, indent=2) + "\n", encoding="utf-8")
        save()
        checkpoint_exporter(run_root)
        torch, EvalConfig, Evaluator, load_tasks = source_audit_runtime()
        if torch.cuda.device_count() != 0:
            raise ValueError("Select Accelerator None for the source audit")
        if (sha_file(data_dir / "tasks.jsonl") != plan["tasks_sha256"]
                or sha_file(data_dir / "HARNESS_README.md") != plan["guide_sha256"]):
            raise ValueError("Organizer input differs from the frozen contract")
        if (set(problem) != {"instance_id", "repo", "base_commit", "problem_statement", "hints_text"}
                or manifest["reference_fields_included"] is not False
                or (manifest["task_id"], manifest["repo"], manifest["base_commit"]) !=
                   (problem["instance_id"], problem["repo"], problem["base_commit"])):
            raise ValueError("Mobile input identity differs or includes reference fields")
        packages = {name: importlib.metadata.version(name) for name in plan["expected_core_packages"]}
        report["core_packages"] = packages
        if packages != plan["expected_core_packages"]:
            raise ValueError("Installed organizer packages differ from the observed contract")
        task = next(row for row in load_tasks(data_dir / "tasks.jsonl") if row.instance_id == manifest["task_id"])
        if any(getattr(task, name) != value for name, value in problem.items()):
            raise ValueError("Official problem-only task differs from mobile input")
        evaluator_module = inspect.getmodule(Evaluator)
        apis = resolve_organizer_audit_api(evaluator_module)
        for name, function in apis.items():
            source = inspect.getsource(function)
            implementation = inspect.getmodule(function)
            (run_root / (name + "_api.py")).write_text(source, encoding="utf-8")
            report["official_api_provenance"][name] = {
                "implementation_module": getattr(implementation, "__name__", None),
                "declared_function_module": getattr(function, "__module__", None),
                "signature": str(inspect.signature(function)),
                "source_sha256": hashlib.sha256(source.encode()).hexdigest(),
                "resolved_from_actual_verifier_namespace_or_snapshot_public_reexport": True,
            }
        if report["official_api_provenance"]["verify_task"]["source_sha256"] != plan["expected_verify_task_sha256"]:
            raise ValueError("Official verifier differs from the recorded fixture version")
        snapshot, base, task_patch = apis["resolve_task_snapshot_paths"](
            data_dir / "snapshots", task.instance_id, task.repo)
        report["official_snapshot_sha256"] = {str(path.relative_to(data_dir)): sha_file(path)
                                                for path in (snapshot, base, task_patch) if path is not None}
        report["status"] = "official_API_bound_starting_source_inventory"
        save()
        checkpoint_exporter(run_root)
        config = EvalConfig(tasks_path=data_dir / "tasks.jsonl", snapshots_dir=data_dir / "snapshots",
            results_dir=run_root / "unused_results", submission_dir=run_root / "unused_agent", models={},
            sandbox="subprocess", timeout_seconds=300, wheels_dir=data_dir / "wheels",
            graph_dir=str(data_dir / "graphs"), embeddings_dir=str(data_dir / "embeddings"))
        evaluator = Evaluator(config)

        async def inventory():
            sandbox_id = await apis["sandbox_start"](evaluator.docker)
            try:
                # Same argument shape as the retained official verify_task source.
                await asyncio.to_thread(apis["extract_snapshot"], evaluator.docker, sandbox_id, snapshot,
                                        patch_path=task_patch, base_snapshot_path=base)
                result = await apis["sandbox_exec"](evaluator.docker, sandbox_id,
                    "cd /workspace && python -c " + shlex.quote(INVENTORY_PROGRAM))
                (run_root / "inventory_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
                (run_root / "inventory_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
                report["inventory_exit_code"] = result.exit_code
                if result.exit_code != 0:
                    raise RuntimeError("Official snapshot inventory failed; inspect retained stdout/stderr")
                records = [line[len("MOBILE_SOURCE_INVENTORY="):] for line in (result.stdout or "").splitlines()
                           if line.startswith("MOBILE_SOURCE_INVENTORY=")]
                if len(records) != 1:
                    raise ValueError("Missing or ambiguous official source inventory")
                actual = json.loads(records[0])
                (run_root / "official_source_inventory.json").write_text(json.dumps(actual, indent=2) + "\n", encoding="utf-8")
                report["official_inventory_sha256"] = sha_file(run_root / "official_source_inventory.json")
                report["comparison"] = compare_inventories(manifest["workspace_sha256"],
                                                           {name: row["sha256"] for name, row in actual["files"].items()})
            finally:
                await apis["sandbox_stop"](evaluator.docker, sandbox_id)
                report["sandbox_cleanup_completed"] = True
                save()

        with ThreadPoolExecutor(max_workers=1) as pool:
            pool.submit(lambda: asyncio.run(inventory())).result()
        report["status"] = "completed"
        save()
        checkpoint_exporter(run_root)
        return report
    except Exception as error:
        report.update(status="audit_exception", error_type=type(error).__name__, error=str(error))
        (run_root / "audit_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        checkpoint_exporter(run_root)
        raise


In [ ]:
PLAN = json.loads('{"scope": "public_development_source_equivalence_not_agent_trial", "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "guide_sha256": "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb", "expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "mobile_input_manifest": {"format": "native_repository_input_v1", "scope": "public_development_input_preparation_not_inference_or_correctness", "task_sha256": "5e7693ff765501da7769c7742633c2a9b77cd20ba833970da45cf5b5c2f01bc4", "task_id": "rich_3067", "repo": "Textualize/rich", "base_commit": "1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d", "source_snapshot_sha256": "8e8f4518217efbc4c7042db995be4093872d925ce7a0edd20e53ad0a31fd5f5a", "workspace_sha256": {".coveragerc": "514783e86d8a98320f8c0ca3ceaa348a6d23c286abf3632e9e69d8a6b38e787b", ".faq/FAQ.md": "2177190440757d3d629928369fa1ec40c63a37f0451ae908c9cc0e4281c8a402", ".faq/suggest.md": "8b72f9a8377e2e1cc84d5ea45162b0411ce1b5c77b9fe5c8b2d164a24951ac29", ".github/FUNDING.yml": "ce6201a5eb9ca2c0ae16322359296ba0f0670424676989e3104bb7011859e238", ".github/ISSUE_TEMPLATE/bug_report.md": "58c0d10681f8951014b1572f8fc6549a59cea3fae5dabfe158718817e8fd78bf", ".github/ISSUE_TEMPLATE/feature_request.md": "0af085a67ab80ecefcfd8e897ff4a515669d5511435ea94634e93488ffd64604", ".github/dependabot.yml": "7e9f02d1bf80a06f59ee643005296a47a0b6ffe9751482cfb3a043a415550e61", ".github/pull_request_template.md": "fbe982f35a8456f18fdaae303e9ee8ea4a5d87dee1c6aab985a81e76b306ce47", ".github/workflows/codeql.yml": "de97644bafb76b9466df8e2b87448be95af6549768c5a45b2bd1d5f2060e6aeb", ".github/workflows/codespell.yml": "edd175f9f9b5a9bc8334e591d9815349e163d36a4566d6882665a4012520ca9e", ".github/workflows/comment.yml": "29fbd133bd9a0661c2b86536799058ba1284d8af632603c2fe234d3482a62f4b", ".github/workflows/newissue.yml": "5489146a1f7b037fdc1e93fd7669808bb89e519ecf9144b4b88021fbfd4ad8eb", ".github/workflows/pythonpackage.yml": "4db31583ed4290afcbf58f799524d7ff0fa63cccf466a4d66f536717394cb898", ".github/workflows/readmechanged.yml": "62b1d6d658e5ef79ee3350dd141d28e9fd1565571f1e9b6f74a703607e38a23f", ".gitignore": "089e9c2c7156e17f6ce5b4943ed2ed5be8ac0f2b64890263008785c52384ef7f", ".pre-commit-config.yaml": "c72b2021ffff6eae3448d5cf7b06e05c650506fc9a8bb4363d5bb8717075bab9", ".readthedocs.yml": "23ff0011b0d86485a25b129841a3b51c2c392ec8b86ac61782db4fcb338b5d5c", "CHANGELOG.md": "b4fdb5e048a0c7eb40be696d3841241ae39295a2b180782743e14b28367a60d7", "CODE_OF_CONDUCT.md": "f723c6d737f218239bf20448515b5af231b0af8400568e6ca515290b8d69d5df", "CONTRIBUTING.md": "9534ac04a08fd68c4b58f9e8716c119387e60b452406aeffe7e72b7a5ffea2dc", "CONTRIBUTORS.md": "98e61f1f5875810df6a797e32c6086ed47f7b47d9a86d6bd3ca6fc236e57cd9e", "FAQ.md": "61d9322ff03692536eb9216e905c7c4a978e5f1bad4f20aff0f63df4cdbe62ef", "LICENSE": "deed7c17a4318158190a3ea239cc879a5a50271cebb98ae7025f48fbe58dca15", "Makefile": "a759aa31b230fcae46832b7fe8436c0dda0738b34bb583fe28fe79616746ab6c", "README.cn.md": "ac1ecffa9a6bd64ebb153d2bdedd6a7e394c361c7eb34e1c7571ef1da35d8b13", "README.de-ch.md": "b7c6e521c69b4b1b72b715b34feaed98305c494f9b41f2462ec16f18af636754", "README.de.md": "6293d786c635f44c69f3874f07f3f531302c6e5b10cf4c6fe60c9b7fadde0eba", "README.es.md": "54f4af6d4f5cac0c369690ad816d66059d948a7a99eda868e8f9937f2ce385b5", "README.fa.md": "456a87b71c3286913b054dacd1f86a7f4ab30ca3f8a8b9c5d21d80e6df09b42c", "README.fr.md": "c830f2850a69126c4cf5ecaec7c822f5e8142cfdb0c7124b3ebb528c170fefff", "README.hi.md": "ceb7408fb0bdbd4e70c79bf3f3aa0b14dfecc18f939bd7cfa869b3054ed6ce41", "README.id.md": "8f322d40c129ecfdce937565fb3a63c63609bb1d3727b0341bfcc3fac241468c", "README.it.md": "9e515741df9202fc0847231f86cca56b45c9408cffb86adfacd884255d1b0505", "README.ja.md": "b07dd318fd9d42cf41b49c5333be91ee42a794fe057acaf045726aaf68cdf79a", "README.kr.md": "6a564874071edc5caac6ebfe1b7c8afc0a6dc8f0f160ef75328e4186ad18e470", "README.md": "5d4fb427e277b1231b663b4871a8b05df0888445f495608becf1e098780c0fda", "README.pl.md": "8a79eb52c11dd21da0380e7ec419fd215bb5894fcd8543780ced5a71b15cc72e", "README.pt-br.md": "cca2be26f89bafd8cb67d83a94fc441cb64038cc713af8bab18ea537646ff06e", "README.ru.md": "dc07f021fe9773da4d2317e1a1f50a4563ce47f46bd7a2d5c9254824b6b75352", "README.sv.md": "2f0eb201c91d3f506c3e10aefc67b48f090432a6fd5e9e0adeaa973653c1f283", "README.tr.md": "929319741b5ca1ea9292904efc7490444d1b00ec47b9c1def55718195b02899c", "README.zh-tw.md": "2640cd9d2a5c1c04d07e1f7227d8498399400daf6c6fcc2f528e7bfad0673f9d", "SECURITY.md": "2a5cd4a7ec86efa16095818b732889533e0f0791fc1bf69d90582eb9b98773c8", "assets/logo.ai": "f3cc38a0ca73602cca9ff71b5d4d212aa61b378a05746defa291c0bf39ac49b7", "assets/logo.svg": "3c05551768405a5550f3e5a5971b5f0e13fcb137a96f63c82d270042db03d4ec", "assets/logo.txt": "7bc6d84ba3a193c365998e0c9b47f457747a1786f07414e11ad72aade5fe9905", "asv.conf.json": "145e1d5a5c31ad6c9e37cc9b64c708e40503e21a66cf68d04f24c6cecd4c94f4", "asvhashfile": "99aabe48a0162888bdc17c8230af41805470ac80f02a2f1d75a7b320bfd37bb5", "benchmarks/README.md": "2b59660660ff7ece615c2f58e9e1ba89ba88827f635176a14599b3376eab27cd", "benchmarks/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "benchmarks/benchmarks.py": "d8bf76728b4c122b94431f09e2c0294d63b374bf0e3bfeac90197b93482781fe", "benchmarks/results/benchmarks.json": "2535a6c17c08142395743c63ebaff02a87ba99e7d778ece1d81c4a90d331d6bc", "benchmarks/results/darrenburns-2022-mbp/008854c4-virtualenv-py3.10.json": "fa6cf1f9459ed2bf12701e22920263615d92c3cfa72ea92627685bbffc81df36", "benchmarks/results/darrenburns-2022-mbp/03392a1b-virtualenv-py3.10-setuptools59.2.0.json": "c740c264e0b21d88d1cbe7f70191dc4a75be64e302ba94b81e6c9be3edb896ef", "benchmarks/results/darrenburns-2022-mbp/038e22eb-virtualenv-py3.10.json": "4a6a94686cef816254cb003a0760c505f438ce0e113beccae12acd1403769470", "benchmarks/results/darrenburns-2022-mbp/03a52134-virtualenv-py3.10.json": "021f57de58f58f81580e62e653c6e8a855233c215d8f73a4a26dbaa3bac19c1f", "benchmarks/results/darrenburns-2022-mbp/06922006-virtualenv-py3.10.json": "54352828ee91239511bc2252966c6e5d4d46ddd4f9f904e4b5706f3441ee0ec3", "benchmarks/results/darrenburns-2022-mbp/06aa1271-virtualenv-py3.10-setuptools59.2.0.json": "e61a1616ff43791df1e6dfd4e79803e4ff26da4a6e91ff70720a5c9c28fde342", "benchmarks/results/darrenburns-2022-mbp/07d51ffc-virtualenv-py3.10.json": "9aa3a30a21f93fa5ea01b7a0dea3081431bb40e83f958e66a6630f5d4905ef9b", "benchmarks/results/darrenburns-2022-mbp/0a3fcb9c-virtualenv-py3.10-setuptools59.2.0.json": "b49739cd6952e84a3d15593756be899a81140d6dc55bd36a1c0d4c72664fda64", "benchmarks/results/darrenburns-2022-mbp/0ac4e308-virtualenv-py3.10-setuptools59.2.0.json": "a8411efc620b2bfca74349e6dd61c37fb2022fab2d6559c44c005c3c51980f4f", "benchmarks/results/darrenburns-2022-mbp/0d2aeb75-virtualenv-py3.10-setuptools59.2.0.json": "0ea803ee6a5cccb5d466c257444d668bf66753a17fc8fd44597a8af7f8963953", "benchmarks/results/darrenburns-2022-mbp/0d69004c-virtualenv-py3.10-setuptools59.2.0.json": "3f69267c25dd1f7d4a55d9585e3b33df28882f18fd1db9755597bb3a1f8e39b0", "benchmarks/results/darrenburns-2022-mbp/0db0cbd0-virtualenv-py3.10.json": "53791329ad1e4ba3b2635afafb7ca0ceb69bafc4db9d089ef530825de7bede43", "benchmarks/results/darrenburns-2022-mbp/0e8df8cd-virtualenv-py3.10-setuptools59.2.0.json": "ba9bdce989498338319093ff1db06a730f9d1e449f0ffe90583ae36d2af311ec", "benchmarks/results/darrenburns-2022-mbp/0fd6bc56-virtualenv-py3.10-setuptools59.2.0.json": "f2ff5d21023b20f6ea146b9c0c8a36105be257f3f2b18258bdfbefc34d60fe8c", "benchmarks/results/darrenburns-2022-mbp/11c00224-virtualenv-py3.10.json": "b23b4edf870b7b045000511b4c5d8d0d7f7557a47f01cb2e48dadabf4aed0f0f", "benchmarks/results/darrenburns-2022-mbp/11c305e1-virtualenv-py3.10.json": "122707f7a85bd51ea2d7ef985915cebb77b9ee67697cdff6df666d30f22ff684", "benchmarks/results/darrenburns-2022-mbp/1442dd77-virtualenv-py3.10.json": "a3f2dc3c93da314d4f9e2178b2b4dde69058eaada2ed2bb6f5b1f033ca6b59e3", "benchmarks/results/darrenburns-2022-mbp/15623c5a-virtualenv-py3.10.json": "f9f2d13407cfc6aa373c44e6a2fa3c8bbba4c2cb81f9b5db6e5f92a5695ad7d0", "benchmarks/results/darrenburns-2022-mbp/177958c5-virtualenv-py3.10-setuptools59.2.0.json": "941bb5907089cd002614901cf0ee6f9be459289c040aa16e577bf0ce00ea1d00", "benchmarks/results/darrenburns-2022-mbp/189a2a3f-virtualenv-py3.10-setuptools59.2.0.json": "51b264b7dac1dbee440670b2d5d1814a8fd25f8230480b1c4465c2e1d94c3644", "benchmarks/results/darrenburns-2022-mbp/19e26c94-virtualenv-py3.10-setuptools59.2.0.json": "9b3b1d28be8f55c83d3f9b4e6d9fc519672490d0c5b3682b4863ce77c8fdea27", "benchmarks/results/darrenburns-2022-mbp/1cdcd1ae-virtualenv-py3.10.json": "a8a3f5e2275d00869d41c9f9273a1d8df034663658defb435ab6cb1bf23429e1", "benchmarks/results/darrenburns-2022-mbp/1daa1771-virtualenv-py3.10-setuptools59.2.0.json": "c20576489b54615d53e24c0ea5d3da9e9054a1c481b741faf662452f9d74ea36", "benchmarks/results/darrenburns-2022-mbp/1f3f7f1e-virtualenv-py3.10.json": "098f98d2343669c082678f5548224bd6baf445c4627af1b53e1ce2892202e72c", "benchmarks/results/darrenburns-2022-mbp/1ffbd443-virtualenv-py3.10.json": "991d8daf52db1b9ded50f6bc00578a30c6c6d4d1a5c8148ef62e95dfaf5d1ef8", "benchmarks/results/darrenburns-2022-mbp/20024635-virtualenv-py3.10.json": "8afa44803506ad139482e257ba27437c8addd3c6834c1afcf988e5c103edaf55", "benchmarks/results/darrenburns-2022-mbp/21432b4c-virtualenv-py3.10-setuptools59.2.0.json": "e092c90da7d73543f88112153a6207f8891705e35c6f6206d00d78a5660cb493", "benchmarks/results/darrenburns-2022-mbp/2356d7c0-virtualenv-py3.10.json": "145276e4934655060039336f5696e3186fe8b2e223c3effcad4a449ab7310349", "benchmarks/results/darrenburns-2022-mbp/23aa7177-virtualenv-py3.10.json": "b5e06906293f44a2eaac00a6afe658b57a836437a6cb6db7642afbcfe75ecc2a", "benchmarks/results/darrenburns-2022-mbp/24743154-virtualenv-py3.10.json": "3c6e5807300a84ae02779caca180945167d256fa92f75404fac61434231528ce", "benchmarks/results/darrenburns-2022-mbp/25a1bf06-virtualenv-py3.10.json": "c36bb6656b88eac5ed68d086bc8207a8aea6993bfad4113978b1dbfb705a6e12", "benchmarks/results/darrenburns-2022-mbp/26fe4667-virtualenv-py3.10-setuptools59.2.0.json": "0feedb0d6bf3c65560b3b5ed31a440cd9ff2d0f48c4ebca20993fb7958461255", "benchmarks/results/darrenburns-2022-mbp/27ab1732-virtualenv-py3.10-setuptools59.2.0.json": "d3af4200d078133deef3d39478f6287efc2456f2e1fca82e7124d6a22ac2f676", "benchmarks/results/darrenburns-2022-mbp/2aea8526-virtualenv-py3.10-setuptools59.2.0.json": "5389247559a4be8dbb0b609776ce328643119404be0ea2d2592d0df77946a6f4", "benchmarks/results/darrenburns-2022-mbp/2ba277ac-virtualenv-py3.10.json": "a6062873d3b0f21654c1eb97c97cf7aff4331cde091f10c4750d973f50bb9407", "benchmarks/results/darrenburns-2022-mbp/2c93dce9-virtualenv-py3.10-setuptools59.2.0.json": "c43becf227d16da4b24daa6f268c91b6ccfc3a15a168d7a0f85ea300a100d2f4", "benchmarks/results/darrenburns-2022-mbp/2d3152a2-virtualenv-py3.10-setuptools59.2.0.json": "98a7efc1e9537b77db5ee5b7b6489f2872438018b0cc67f10f76b3649c4c91fd", "benchmarks/results/darrenburns-2022-mbp/2d3ec69f-virtualenv-py3.10.json": "9b0bb1b1de9a149f6529540293c0e4bdad3c1ee135eb74020725b1faeb82d353", "benchmarks/results/darrenburns-2022-mbp/2ea7e586-virtualenv-py3.10-setuptools59.2.0.json": "00623f0d35bb0e0eef9aba54f5d802a8580f3081e60b31790e074bdba95bff70", "benchmarks/results/darrenburns-2022-mbp/2ea7e586-virtualenv-py3.10.json": "503f5f64977ee4b10f431a21d01bff4fab9d3f86d13fbb723400f53964f93e51", "benchmarks/results/darrenburns-2022-mbp/30498f59-virtualenv-py3.10-setuptools59.2.0.json": "2917a68171765f1e17dbac07059421ce14ec5cb4c19f288e35297e00102bcdf5", "benchmarks/results/darrenburns-2022-mbp/3473658d-virtualenv-py3.10.json": "9fb965df6315bb23ecb3f36a2631008371a8fdf3c00daa2a8f4cd65d0df08c94", "benchmarks/results/darrenburns-2022-mbp/36efcb5a-virtualenv-py3.10.json": "9e52f206a56e919cf8e045e0df390d7bfa2d8a365239146d339b9ec4d688e8bb", "benchmarks/results/darrenburns-2022-mbp/3827b4ae-virtualenv-py3.10.json": "085ff68f1691c94c183a624e872adcbff95b16252f3fc26a4a28241079cfcdd1", "benchmarks/results/darrenburns-2022-mbp/3be88c08-virtualenv-py3.10-setuptools59.2.0.json": "4951034095c44ea90571f70a6351178a7d00c425ed5e6d0ec8c59b198a7199f9", "benchmarks/results/darrenburns-2022-mbp/3db6396a-virtualenv-py3.10.json": "cb6d265cfcffe2bfba5b515b3d529f615684fb37899a780164af2e89af249d4b", "benchmarks/results/darrenburns-2022-mbp/3f7d3e4e-virtualenv-py3.10-setuptools59.2.0.json": "c3debbe17546f11c37262a8c9c5336410ae8c8973ee29b7d1faad5a99c266297", "benchmarks/results/darrenburns-2022-mbp/3f7d3e4e-virtualenv-py3.10.json": "589dd9f774472a5b65eb6b46bbd5fb6bf5573d23711bbb1f9e09fa1cd3c8b3a7", "benchmarks/results/darrenburns-2022-mbp/4020d5a9-virtualenv-py3.10.json": "67f4ea988b29c10d41b5e4f98c60e04cd4fb51f9242241e35eceec79f757111e", "benchmarks/results/darrenburns-2022-mbp/41279bca-virtualenv-py3.10-setuptools59.2.0.json": "9f6efc0fe784069e6ef0c17584a9fe50e7e08aba21909e3f15c0f98e6700a9b4", "benchmarks/results/darrenburns-2022-mbp/416033ff-virtualenv-py3.10-setuptools59.2.0.json": "0531470b4b73d2b53c5ff206caf909f3261e209c8b7152d84b25bc1f31ec6f61", "benchmarks/results/darrenburns-2022-mbp/43a26c0a-virtualenv-py3.10.json": "10f5c8b63e7b4e8a8a452cc71725f702e743762f5a6e39f1db4bc10679de9c2c", "benchmarks/results/darrenburns-2022-mbp/43d4c4e5-virtualenv-py3.10-setuptools59.2.0.json": "bbe0cea11348392840550a8887d74b664982fb60b6d8ade51c5159db5170f88d", "benchmarks/results/darrenburns-2022-mbp/43d4c4e5-virtualenv-py3.10.json": "6dbdd8e9481a38147113a231483f026b33a749315c1bfba86f83957fa6a6f43f", "benchmarks/results/darrenburns-2022-mbp/44f54dd8-virtualenv-py3.10.json": "a4f57e8290eaa318d838352e26e10c9f1e04f6884abaa30b0a90a8f248635054", "benchmarks/results/darrenburns-2022-mbp/464e4e33-virtualenv-py3.10-setuptools59.2.0.json": "b3924629f47f3200e9ba2e6de2ce439ca11042add549e659cec28b92d6b36869", "benchmarks/results/darrenburns-2022-mbp/489fafc6-virtualenv-py3.10-setuptools59.2.0.json": "578ca28d612dd5fa5a5e1e346c89516ea500989cc24f0e53ccf5f26b2f10ad31", "benchmarks/results/darrenburns-2022-mbp/48da2791-virtualenv-py3.10.json": "9d77d8635fb8694f813fb8c5d6d1ebf41e3e04f45dad6a47534187dac3ff06b1", "benchmarks/results/darrenburns-2022-mbp/4b123ddf-virtualenv-py3.10.json": "ba5d47503be5c68134536bd1e8c519db6fadee899ebbac1114f855c4ab789408", "benchmarks/results/darrenburns-2022-mbp/4b3b6531-virtualenv-py3.10-setuptools59.2.0.json": "a61c4aa3a5a8cdf20b04f6f900bde7e037ed472912fa1f26eb65d59070ee0409", "benchmarks/results/darrenburns-2022-mbp/4bf3f19c-virtualenv-py3.10.json": "000319ff3d599cd9d60b1abad68937b1ad0f8ae29869e66341a9cdb11f9cc2c4", "benchmarks/results/darrenburns-2022-mbp/4d6a6d88-virtualenv-py3.10.json": "e5b98538ba2e4bee02963b50e8ce5856b4b3de35707c82212c16d966cfda435a", "benchmarks/results/darrenburns-2022-mbp/4dc1d4cb-virtualenv-py3.10-setuptools59.2.0.json": "0b96ab156e6c6b9b11b5d4789bf734e0cd072e939903d8e88f4d755db70b75d8", "benchmarks/results/darrenburns-2022-mbp/4f8908a6-virtualenv-py3.10-setuptools59.2.0.json": "e516be5fcbc97b40ce908cc64115606461cb90703ecee6484b44c0b330f81ac9", "benchmarks/results/darrenburns-2022-mbp/52d159aa-virtualenv-py3.10-setuptools59.2.0.json": "a58beab9270b8b1a8329a9365c7800723c209ac1f134b87ba18e807cff268222", "benchmarks/results/darrenburns-2022-mbp/52d159aa-virtualenv-py3.10.json": "857bcfde665601faa933e8e0cc459b60ab29aba4bb5b5f0d9e9c9f2a0d6c7734", "benchmarks/results/darrenburns-2022-mbp/53cda574-virtualenv-py3.10-setuptools59.2.0.json": "5b2586e7631623b9d1e003681338d28df43200af02de0beaa63554093540f2f4", "benchmarks/results/darrenburns-2022-mbp/53d9eeaf-virtualenv-py3.10.json": "c43ce975697828d21940096b59ecea3ca49f9cf788f156ae476d102621956378", "benchmarks/results/darrenburns-2022-mbp/550d3911-virtualenv-py3.10.json": "270e39b60080c566876236fd3b7983a27084f4b630750d4c597c7f1b367fb897", "benchmarks/results/darrenburns-2022-mbp/55e11902-virtualenv-py3.10.json": "899b5e6b63910c46e1bc126657bdd30e3e972b1eaf5b19d0dccf89ae2cdf176e", "benchmarks/results/darrenburns-2022-mbp/573125e9-virtualenv-py3.10.json": "6a8ea3c95c5a5d94e5262d18b859b12101affcfed6beadb3defcf62c8886a60f", "benchmarks/results/darrenburns-2022-mbp/579a29c8-virtualenv-py3.10.json": "671cc17bb25c63582e08c14f6415718acb8118902f8486cfcf2daeb4d27bbb61", "benchmarks/results/darrenburns-2022-mbp/588f0331-virtualenv-py3.10-setuptools59.2.0.json": "ef6fc003a67248ac01f55bfde47f165fed9096143df9625a54f40ba13358564a", "benchmarks/results/darrenburns-2022-mbp/58bfa48f-virtualenv-py3.10-setuptools59.2.0.json": "6cf7e9f260b202cf7f3d55c2f15143ce6cfe4bba66c94b6ea02532c2c03392a3", "benchmarks/results/darrenburns-2022-mbp/5f021978-virtualenv-py3.10-setuptools59.2.0.json": "af8fce0ac3d3f84202329c48818c879ad4e36bee41c40ca68cbf6007cd9e5b7a", "benchmarks/results/darrenburns-2022-mbp/5f021978-virtualenv-py3.10.json": "05f0cb907e719a53efadc82c9fdaaf0d40b0604a6488b71efb18023f967378f2", "benchmarks/results/darrenburns-2022-mbp/5f03e3ba-virtualenv-py3.10.json": "22e358591edbd72aa48d1ef5614d6336d437bd4e8fa9f4aa572b7f588372e7e6", "benchmarks/results/darrenburns-2022-mbp/5f55063b-virtualenv-py3.10.json": "044633b3af3682ce7f64f8d16a5ba9666f3f94bbdef66f699bf6b8d9e849ea78", "benchmarks/results/darrenburns-2022-mbp/5f82274a-virtualenv-py3.10-setuptools59.2.0.json": "d29a50fb489a538f75ec9b881cdd42f8ed4e6f55a6ce8304d184a410b2511de0", "benchmarks/results/darrenburns-2022-mbp/5fafb92f-virtualenv-py3.10-setuptools59.2.0.json": "23805dfdb4cfcfd481bdbef369a7b49217a3d347c260fdb1f180768d367f8398", "benchmarks/results/darrenburns-2022-mbp/60dadaf2-virtualenv-py3.10-setuptools59.2.0.json": "4183a0e42c3faf2e553cc9ca17a662aa464eedcbd7c80eed87c656058a02ac66", "benchmarks/results/darrenburns-2022-mbp/64471afc-virtualenv-py3.10-setuptools59.2.0.json": "20d3fd53d145b2d652f595b59ad5b3f7c811d7b129f1dc5496512a83d0425190", "benchmarks/results/darrenburns-2022-mbp/646d933d-virtualenv-py3.10-setuptools59.2.0.json": "c57371a8305c18ecbf26c857725ebb808ef721829aee38b2ca65ae99ee1e6f8a", "benchmarks/results/darrenburns-2022-mbp/64755d41-virtualenv-py3.10.json": "5f4c7d2d00dfc6fb4b63f985eb8d08e17c47a5106612427b475501f6537dc3fb", "benchmarks/results/darrenburns-2022-mbp/656b7a18-virtualenv-py3.10-setuptools59.2.0.json": "4f8120d7f540546ea4beecba8fb9584e46bbe40a4bce29e600aef44dc7efbef2", "benchmarks/results/darrenburns-2022-mbp/666d0cf2-virtualenv-py3.10.json": "5fa49ba56228c9473991686e8d0445c2078a245f25c4df7af36997b4256e6543", "benchmarks/results/darrenburns-2022-mbp/690507d4-virtualenv-py3.10.json": "8078099b1c4796dc12eb0b792d6349acd77da2fb6998e67bdc5417b0238262da", "benchmarks/results/darrenburns-2022-mbp/6d7ba589-virtualenv-py3.10.json": "db51d35f81e1c3d3baf827dd81d9f3e954d6d2c8a22ff9233652ae2c55472ca8", "benchmarks/results/darrenburns-2022-mbp/71135d19-virtualenv-py3.10-setuptools59.2.0.json": "1e2d48fb20f0a22696fcf8f730051ae13146109a0887a95a3c6e077b73382cca", "benchmarks/results/darrenburns-2022-mbp/7441bf27-virtualenv-py3.10.json": "70168d014ff1e7aed47087080a1f64d14cee94ea0a32e6f099ef686f1a0978df", "benchmarks/results/darrenburns-2022-mbp/76620730-virtualenv-py3.10-setuptools59.2.0.json": "d705299b24ce0e187aea8f2ab92c77758b09ee2fd0fc3e111737bd091ec61e12", "benchmarks/results/darrenburns-2022-mbp/79ea1c1d-virtualenv-py3.10-setuptools59.2.0.json": "f981542f84397ffa8cdb5ee840dc8912c19d8da319c459d35abbc849beba2cb5", "benchmarks/results/darrenburns-2022-mbp/7bad81d5-virtualenv-py3.10-setuptools59.2.0.json": "73187bc0db5be602b0322b57a2332fffade0c59ec92f245f3952c0f715c60cbf", "benchmarks/results/darrenburns-2022-mbp/7d00fa83-virtualenv-py3.10.json": "032ac3657ffaa199bce1bd7810ec893e2f05700701420cfaa0f8f2eb57649298", "benchmarks/results/darrenburns-2022-mbp/7d02d29b-virtualenv-py3.10.json": "5a55b44a4392d6eb7521c8ce6fe0db7455d12d5ec796162ae31340f3aedee3f7", "benchmarks/results/darrenburns-2022-mbp/7e4a2db4-virtualenv-py3.10.json": "547876a84b288690c8797e42acdae97cdfa6fe36e7fb48d1c82dfe89e405c2f8", "benchmarks/results/darrenburns-2022-mbp/7edd619f-virtualenv-py3.10-setuptools59.2.0.json": "6c826b4a90257c671926c52f525036686d29e15b6730715f38e083272900e458", "benchmarks/results/darrenburns-2022-mbp/7ef7ffee-virtualenv-py3.10.json": "89d2271c1b19aaeb0c6354e7f9f8209afdf86a5a780b103a4d0875124d90553a", "benchmarks/results/darrenburns-2022-mbp/83756d62-virtualenv-py3.10-setuptools59.2.0.json": "48c02ac2f33f69d089ffc52b53cd4687c9b6a4fe3353b260e658ecb0013469fe", "benchmarks/results/darrenburns-2022-mbp/837b6d7e-virtualenv-py3.10.json": "405c81f6375d63bd664f9fbe832cf54d119a52c869768d5feb220ba73f39fcaf", "benchmarks/results/darrenburns-2022-mbp/877c53d9-virtualenv-py3.10-setuptools59.2.0.json": "afab1850fc30f8befb09f0e4fb289e33f13defbb8284e036105ddb37498af687", "benchmarks/results/darrenburns-2022-mbp/88b07b3e-virtualenv-py3.10.json": "60e229e5cf8c496fe0971d26e7ba7feed4143f32005c9602be8afc4988a59702", "benchmarks/results/darrenburns-2022-mbp/8a7f5d82-virtualenv-py3.10-setuptools59.2.0.json": "15b7a91de8ce034b5815fd6bc21db0dafb27d6cd8f64ce3344ce3bad434b8941", "benchmarks/results/darrenburns-2022-mbp/8a7f5d82-virtualenv-py3.10.json": "7dfbba749443cfc2e2fc9a730563ef93f486d48d5364cda12cc08f0d1697c0c1", "benchmarks/results/darrenburns-2022-mbp/8b185610-virtualenv-py3.10.json": "64369bea89654a8fcbe22779dcd0a03553e80d5e5d947276e9e55e1785b14ab9", "benchmarks/results/darrenburns-2022-mbp/8b47f338-virtualenv-py3.10.json": "8300935bed4e9709d8e761324c5c29f1a50eb1b390144883a7ca665091878eea", "benchmarks/results/darrenburns-2022-mbp/8c3e6be4-virtualenv-py3.10-setuptools59.2.0.json": "7b74e6ec32f4a5ce011d19dda94956594a71e3ef62d118061dcb0757147ccaec", "benchmarks/results/darrenburns-2022-mbp/8e649fea-virtualenv-py3.10-setuptools59.2.0.json": "99e2fed99d642c49cf5d181c8aa210e5097aab9ba303f13bf667453455578224", "benchmarks/results/darrenburns-2022-mbp/911d305f-virtualenv-py3.10.json": "8d700d8cb2f0cc0d1efd53e04ba2c5d7aad431ee42ef4dc6917303255afec2e1", "benchmarks/results/darrenburns-2022-mbp/932e26b6-virtualenv-py3.10.json": "294762aba667163e120c32f839bb18450158d530ea6b6ef0d2cf3fe4b43faaf3", "benchmarks/results/darrenburns-2022-mbp/949e1f72-virtualenv-py3.10.json": "14c4dc7270409f7d76c3e43d23466be0d396c87a5ae0ab871ce803a009fe0788", "benchmarks/results/darrenburns-2022-mbp/95d8bf98-virtualenv-py3.10.json": "eb51d550ad688ad16ccc19f310fdcb2a7a2412b419b148c52ef4b215fc36f790", "benchmarks/results/darrenburns-2022-mbp/96ea5fed-virtualenv-py3.10.json": "016b36b3cff344c80aec4b93d48ddc883da6db8db86a7289df41453a04e81f0b", "benchmarks/results/darrenburns-2022-mbp/972dedff-virtualenv-py3.10-setuptools59.2.0.json": "e6ef0f688a6ce157fd243c0f67292114d25b2a9f96982249800eba5c241572eb", "benchmarks/results/darrenburns-2022-mbp/972dedff-virtualenv-py3.10.json": "13b592c0bb5e3336109688bbdf9828321bca2b6aece39dbf6cf9423bd0a5282b", "benchmarks/results/darrenburns-2022-mbp/99831099-virtualenv-py3.10.json": "7451e3e3176dde2c8e11b23b4334a79093381897e861d854ea80b140be0bb919", "benchmarks/results/darrenburns-2022-mbp/9a4fbf83-virtualenv-py3.10.json": "2bd543b691f65a4960075a5dbb7f8e99b7d52c45bb4cf58492bd2e5ce1f56016", "benchmarks/results/darrenburns-2022-mbp/9abc0292-virtualenv-py3.10.json": "fdf09897ba5c5626dd82c8589351da1b0b66b7642cc85377a96756bd2f47cb10", "benchmarks/results/darrenburns-2022-mbp/9bfb6190-virtualenv-py3.10-setuptools59.2.0.json": "ce5c2b5d43818e26f3a07c0fb330b7b80608107bf151ae1c13666181a6861a33", "benchmarks/results/darrenburns-2022-mbp/9f2a426e-virtualenv-py3.10.json": "1c3482f28a645bdd7f8b3514b5e044926f05a3b86762c52c9c9835403b70a171", "benchmarks/results/darrenburns-2022-mbp/a27a3ee2-virtualenv-py3.10.json": "16a5038d315585c1352a1fbfaaa4f8bb985a8a086bc47a835c66fe142da31884", "benchmarks/results/darrenburns-2022-mbp/a2f6688e-virtualenv-py3.10-setuptools59.2.0.json": "93b605e57cbaa8620cf3365762d01dcc752147478b9d025d4000169c08747f0f", "benchmarks/results/darrenburns-2022-mbp/a6d1d784-virtualenv-py3.10.json": "d8ca1a0e62f4e679dbec3d040d06cbdeaea9d1c241fc7e36eed99420a0067ccc", "benchmarks/results/darrenburns-2022-mbp/a6ea9890-virtualenv-py3.10-setuptools59.2.0.json": "6b7e9714fe42df6711b171b96c7f043a791d5050442ac4872881572f0f75b910", "benchmarks/results/darrenburns-2022-mbp/a81230bc-virtualenv-py3.10-setuptools59.2.0.json": "d9a8a46f924b682be6066917385259fb7662a2e8aef923ec2e738aabe68d802f", "benchmarks/results/darrenburns-2022-mbp/a81230bc-virtualenv-py3.10.json": "9b16b37a08c5e4457495413cdaae2fa9fa7aca3748195b043b647d41c94cda0c", "benchmarks/results/darrenburns-2022-mbp/a8d2bb20-virtualenv-py3.10-setuptools59.2.0.json": "fe1024303ae400918410c081d3d0a4b50bd9fc7b2e9e817ac34826737d714f89", "benchmarks/results/darrenburns-2022-mbp/aa4546ac-virtualenv-py3.10-setuptools59.2.0.json": "5a21a9fc6302571e917b48be696aa2beef535eedc497fc9be9f074971caf0904", "benchmarks/results/darrenburns-2022-mbp/aa7926c1-virtualenv-py3.10-setuptools59.2.0.json": "988c2cb355ae726a0c9735ab1ee2b7aaf79267ffc8e360114ce7bb43ca5fd84c", "benchmarks/results/darrenburns-2022-mbp/aaea99f7-virtualenv-py3.10.json": "68f77af271b25ec568d8e2724e43c7cdbac135847bf6909ce92f9211b00f5be8", "benchmarks/results/darrenburns-2022-mbp/ac1a33da-virtualenv-py3.10.json": "7ec33e7389e95d4f998bf67211142a9d3fc261d4f1234da07855f4da158c5936", "benchmarks/results/darrenburns-2022-mbp/aca0b60b-virtualenv-py3.10.json": "ca3c5cad3c394ddde08dcd25e98bf02f42ee00946d0e52cfb8a07efc3ad44e3a", "benchmarks/results/darrenburns-2022-mbp/ad6e3dea-virtualenv-py3.10.json": "6d7eb725b5b64d36dc44ab953ea3b33e7c4607a5596650d3a4cd1e10dc74ef0c", "benchmarks/results/darrenburns-2022-mbp/ae5865eb-virtualenv-py3.10-setuptools59.2.0.json": "91f9c0e5bd091021bd98749fb6da49453bd73105d258d1d3c19e017361252bb4", "benchmarks/results/darrenburns-2022-mbp/b15bc18c-virtualenv-py3.10-setuptools59.2.0.json": "9d78e26fc64632ba7485ee0e0336edbcc6cc6a3f03dac7a3b1742e091c9545c0", "benchmarks/results/darrenburns-2022-mbp/b391635e-virtualenv-py3.10.json": "e6261218cb0e9944274f20968e301e9099f045a1df39aa9d3c81067db6c83c0b", "benchmarks/results/darrenburns-2022-mbp/b9e0014a-virtualenv-py3.10-setuptools59.2.0.json": "e8bc878e27fcc3552976bd3b781fbec5caabe2a3d1a238301961f6f52b2cb64c", "benchmarks/results/darrenburns-2022-mbp/b9e0014a-virtualenv-py3.10.json": "1bf59e933fb55044e8de1b595b32ab0b37a585b486c0b2219eb449a8013ecacd", "benchmarks/results/darrenburns-2022-mbp/ba5d0c2c-virtualenv-py3.10.json": "66f2123fd0cc207a481c9ae1876fe550cc2e29650d895f6bd09dc732097b8a7d", "benchmarks/results/darrenburns-2022-mbp/bd34e0a1-virtualenv-py3.10-setuptools59.2.0.json": "3d50c980de72fcead01d28fd7e838aca6c7cfcc183a6f53836dbca72aedbde49", "benchmarks/results/darrenburns-2022-mbp/bd34e0a1-virtualenv-py3.10.json": "1e320d99a944868991875438986236432d2dfb3af9aa5665ee15737fd45f3d39", "benchmarks/results/darrenburns-2022-mbp/bf728dbc-virtualenv-py3.10-setuptools59.2.0.json": "177c26916bdbcdda5f7ccc0b7e23b8b318f98b9634b9223639726ddf942ab2c8", "benchmarks/results/darrenburns-2022-mbp/c24ab497-virtualenv-py3.10.json": "ee46572dd35a0c7ce1ed99d626218fc0e539f91fbb830e14e13e111c68c7a503", "benchmarks/results/darrenburns-2022-mbp/c3d0e358-virtualenv-py3.10-setuptools59.2.0.json": "9cdfde841039a07de975e755a3eace808be5de02502f2ac1de225b32f698990a", "benchmarks/results/darrenburns-2022-mbp/c3d0e358-virtualenv-py3.10.json": "89fa3936b7c3f31161fa268e5f13587556abe43ba64919e9d271a241a407b741", "benchmarks/results/darrenburns-2022-mbp/c3ee3b05-virtualenv-py3.10.json": "111a6c0f6b1d448cda61dda3d2237286749e150c669b9a92063e6d0b86bbd9d9", "benchmarks/results/darrenburns-2022-mbp/c57e1f50-virtualenv-py3.10.json": "c1178e08ac1b6d21c94a8dea9f4bb49698f80e6adf5fdd7c14dcad97b59e352d", "benchmarks/results/darrenburns-2022-mbp/c9afafdd-virtualenv-py3.10.json": "6aa246e82e5b8874877b7a7ad45533f8d5d8031ff3ba4be32a6651bddc8c18c2", "benchmarks/results/darrenburns-2022-mbp/cefafdc1-virtualenv-py3.10.json": "7142075e15067995f110089142d57c682372d56824e22794e74a548e749c8aea", "benchmarks/results/darrenburns-2022-mbp/cf606f0a-virtualenv-py3.10.json": "90a42935fd83a91ae7403d91b8cb9b21b80ca1c7ff723743c700ea3f2e79d880", "benchmarks/results/darrenburns-2022-mbp/d06540a2-virtualenv-py3.10-setuptools59.2.0.json": "38325526ccde2d2dc0ae2477ff9fe913c07e7b5a277ba4617d8d769d14d9a4ec", "benchmarks/results/darrenburns-2022-mbp/d1ea01d0-virtualenv-py3.10.json": "11b46cd3b233fe3b0aaa047fd22282c5b021247f4c0dfc0c2b39b9a4f11b723d", "benchmarks/results/darrenburns-2022-mbp/d6e6a762-virtualenv-py3.10.json": "1e1b60e62270036729a5d75324e9df0e69d9e7a3dc1887a07f6f3839220a1cb5", "benchmarks/results/darrenburns-2022-mbp/d9d59c6e-virtualenv-py3.10-setuptools59.2.0.json": "52fad3582e58ab10ab744350600471c060e975763c20cc8177e47bdbc3a54bdb", "benchmarks/results/darrenburns-2022-mbp/d9d59c6e-virtualenv-py3.10.json": "4e57abd3b2ddbbaecd61c8d21c813787e2842af403fe95ac386de63a71e0251f", "benchmarks/results/darrenburns-2022-mbp/dc3f0623-virtualenv-py3.10-setuptools59.2.0.json": "ea02cb50d255aec6c7f549f3c6d9a146e91772404e6b46cd753912af4571fa97", "benchmarks/results/darrenburns-2022-mbp/e0a1fd30-virtualenv-py3.10.json": "e5aa10faef82db0844644c3fe5f5ead933871bcd17d3c9df4ccd51a0d5a4bb35", "benchmarks/results/darrenburns-2022-mbp/e21ac11a-virtualenv-py3.10-setuptools59.2.0.json": "1b1a3ef8b53fde4bfe119a2865bf626b206e46f79728fb8630f24800a2f9ad90", "benchmarks/results/darrenburns-2022-mbp/e338ab14-virtualenv-py3.10.json": "b722023f84b2b5366334a208b8b09078441dba701fa1b2881d5ef27c4d2af18b", "benchmarks/results/darrenburns-2022-mbp/e34eadb3-virtualenv-py3.10.json": "fa5338bdc0ac32daf5d99f04baf52a4c0f47f1cfd3d07604e22ba1d294636e9e", "benchmarks/results/darrenburns-2022-mbp/e5246436-virtualenv-py3.10.json": "5ab05a2bd17f43db966eebb8d5c67208b9c18388add8ac6b3f647374b61407a0", "benchmarks/results/darrenburns-2022-mbp/e7849495-virtualenv-py3.10.json": "fa873ac1b87f816f5f182b73ea75fcbc10117196c47bd6a673485748656bcccf", "benchmarks/results/darrenburns-2022-mbp/e78acae6-virtualenv-py3.10-setuptools59.2.0.json": "062373d5dc54ffe18351433a1f7ebe1e1f25b569f94a8ed2cd9d4731667edf3e", "benchmarks/results/darrenburns-2022-mbp/e7de32a0-virtualenv-py3.10-setuptools59.2.0.json": "426321fb9857db5228c8fddd67c9fa1a1a1c5c802aa408ad0788bbc533b7217d", "benchmarks/results/darrenburns-2022-mbp/e9e72000-virtualenv-py3.10.json": "57765a0ef74e0f3bbb2ff362d85cde590512374684c9df035e32c25fddbfcc5a", "benchmarks/results/darrenburns-2022-mbp/ea049ffc-virtualenv-py3.10.json": "0ce676c35e4fbbf6fe82ae0446d562c7375766b2ce22c15a227570f6dd828fd8", "benchmarks/results/darrenburns-2022-mbp/ea2ed337-virtualenv-py3.10.json": "81359de11344921c951206151868a53b4b60a0beed8c5b3924c1b7ab4e31a2ac", "benchmarks/results/darrenburns-2022-mbp/eab3fe8e-virtualenv-py3.10-setuptools59.2.0.json": "83e332d5a762be44e5fc30159a2a136fab5ccd477e35f2b55a5490cfbb76fc31", "benchmarks/results/darrenburns-2022-mbp/ecf3d7f1-virtualenv-py3.10.json": "bdcca2474453b19c2b0264cc95e83fe689f90837d0473402cb31feea5c53e622", "benchmarks/results/darrenburns-2022-mbp/edcb6f9e-virtualenv-py3.10-setuptools59.2.0.json": "9ba3ff14e580116f662ef0c98e54735be9b91bfb4c494f5080d233fc2c5777c6", "benchmarks/results/darrenburns-2022-mbp/ef80460f-virtualenv-py3.10.json": "e1798e7ff95dee487ab8dc822ef9fb06b3c2b03d66036c4a67f925fa230502d8", "benchmarks/results/darrenburns-2022-mbp/f2845e12-virtualenv-py3.10-setuptools59.2.0.json": "1511e080abe28cddfbff2f0c38daf1e3d30548653e58324458a5efa3e4ec11cb", "benchmarks/results/darrenburns-2022-mbp/f2af8c9d-virtualenv-py3.10.json": "b6284d9a61b38a5de26435bb0686ea28af83b125b63ca5bebf32249de477f9a2", "benchmarks/results/darrenburns-2022-mbp/f5ed5bde-virtualenv-py3.10-setuptools59.2.0.json": "efd3912033c6d798610922d70bd46ef3c839304e145a0218f4488e9ecbdba2ea", "benchmarks/results/darrenburns-2022-mbp/f82a4ccf-virtualenv-py3.10-setuptools59.2.0.json": "b33d917994f94a016be8582af4d4f8353bb92cb987456c7e4bb2ca1929b23adf", "benchmarks/results/darrenburns-2022-mbp/f84d5dee-virtualenv-py3.10.json": "0df9bb138000be2c8d9bb712a88cbb6ef10c7ffd691db33c3244e9d5cd076031", "benchmarks/results/darrenburns-2022-mbp/machine.json": "ecc637fe12aeeae9bbe05c1c11a326bcea0f570c69cf63ef381af96071414e0e", "benchmarks/snippets.py": "d4a3f66fcd8f5e7e85e9abe27499a3e5278d12bc8ddae9db21810935cc3d5ee5", "docs/Makefile": "e33bf74d59130029ba24168a813112dd923d7044d780ce942db6645d93f56acf", "docs/images/box.svg": "8e4b82375c9d6666e96288b325cb97180b98d75e502890dacd7f24baf3f3b303", "docs/images/svg_export.svg": "edefc74a334a99637ef07b49337d01e534ced06c4477b3c29f2910b41744d9eb", "docs/make.bat": "e7302d938728984369c2a45c22e67016c7472a1d46ebd4fee145261aa4bb7e9d", "docs/requirements.txt": "63dd32066e8dff57f620399a45b04a528a1a1daf8cc13dc54cc8d1d2d7a0c2bd", "docs/source/appendix.rst": "59ff5cd413dd5bdefdcc734c7e170ca9220368499ecc9c6f751fb58a4ce0d224", "docs/source/appendix/box.rst": "dc6270815f24be719dc7f7db7388d6aec0595c79d9cea39e8b69122d80a3631f", "docs/source/appendix/colors.rst": "3a6578ab4dee0539e7c4d20bc34cb6994b71db439524e63f91d4ad55403f6871", "docs/source/columns.rst": "0a97020e592dc8a05fc37fa07fecb088a9ce7fe5277b6aa5d6be39a8b398e982", "docs/source/conf.py": "bc167b95a9a78655f906156b8a61f68dd33ac3644e773e09db86d794ea7f3812", "docs/source/console.rst": "6ceed537251f41539dce49b01e2111ab57f2cd0f57bfa77de09baba367c0c70f", "docs/source/group.rst": "7b0df2dce905934778bc08e04cefd8af4684123b19bec0a5d5a849f367da019e", "docs/source/highlighting.rst": "52e57ff053bc2c34d3eb1ccbe44b57d2e5e4163f21111c21e0ece22342c41d4a", "docs/source/index.rst": "cbe838fc2460287f97d6b55cd35ed57185cf04011f4db98b8397eca1fd650ef5", "docs/source/introduction.rst": "721b0e7fd0f9ace66bd454f93ddafdeda9ee4782bb8ec7a5fa65240e589f21e9", "docs/source/layout.rst": "6e114ac07aa8389171cbd7ef5c74730409b3840b3c730014ed60431cb4ee4b66", "docs/source/live.rst": "f9d9ca8d88e47b82b82f55e80f66d8b761f2caf631433d09e16c1d69dba041d0", "docs/source/logging.rst": "3a06a07eb337c30887ab3a6a194eddf459d200a88960fc63f6b91c9547e70c75", "docs/source/markdown.rst": "593db727cec3eabeb749b652dce36ffb92bf22da6f1dae7bbb7755d8e70b0350", "docs/source/markup.rst": "43bb4ac18b406a64e75a697edabae4974aec60af7ba2aa2b4d7d47e67073d08c", "docs/source/padding.rst": "127ba6a0eb76ae03332da4f0864d9914f9fb0b7addcadfb05e6febbf26c6cef1", "docs/source/panel.rst": "b228d6bed0cd33ffc3d1e414993fcebc972c6962fe273ee65c30419c5d2445db", "docs/source/pretty.rst": "2a9b0a9eca726fbd977c606a0ba508e24de4d144e5e42f0076ec9d832238e950", "docs/source/progress.rst": "c348cb45e90d32b8376b47209438eee12fbef703b4d0f2fb86b49671a87eff88", "docs/source/prompt.rst": "5025efc79868d277dca52604c97ab17db2b7a7d04a3406c7de780062034122f9", "docs/source/protocol.rst": "06748b8d2baa9b2aa3f804d38c9f7e5a3bb542d4b6e004196ad63e1b3ce3e8e8", "docs/source/reference.rst": "76e5f62b8c93f3a1d08ff810b7029d226573d5c48b3b25e64b0ca66afd89cfe6", "docs/source/reference/abc.rst": "6ad328e57e6adb0931f2aaeb761b9f932c774bf6e91d0bebb864b46b8269817a", "docs/source/reference/align.rst": "de2dce261b603fac1be6c752265df0b36d82023986e8d80697ee16f972b9257e", "docs/source/reference/bar.rst": "4ef3716e29db04f174c7231e8e9b857eb329b2ebec7ed3a86ed203168231a212", "docs/source/reference/color.rst": "0a773993d39b0c83504573dff5ef3063107ded8c3979020d05518c03c61741f8", "docs/source/reference/columns.rst": "3eaed2d24a0f803bd3b293db9529042b1f651a3957804fb86886cf515810156a", "docs/source/reference/console.rst": "188458865b7b0cc8f68d12520f22be3796105bb9c0417c615d810fc5aac1208c", "docs/source/reference/emoji.rst": "33083e4d9b93d66dee4a1096561a52fcd7aee1f15f385edc22043ccd88f4241c", "docs/source/reference/highlighter.rst": "c9a45addd3d465c21b67cd5a0eb92d510715f08159c7c08abce088e6389c3a00", "docs/source/reference/init.rst": "0fbe31adc36df243e7860b73af1dce6d0a2722a958b05f49d9b2f7004de2517e", "docs/source/reference/json.rst": "a21c555a3241eaa2cd3f5f6dc382511475356c02a1084bae21d1573ddaa67b8e", "docs/source/reference/layout.rst": "deba198df46e52d21d2dd3ea8f1b2334d2c8fd88e2b5163d8e581a2b1543ff4e", "docs/source/reference/live.rst": "4eef8a14656793a2ed5600ecb2cb68451484ed00daca1428dc00cf38cb8cd9a3", "docs/source/reference/logging.rst": "72e32b08384065f02c69a0a801321d008b0b699395eff0ee65eb18477df80470", "docs/source/reference/markdown.rst": "b7f9c9451163782f45edd5be45ce85caad60e258844eef9c5a8fd9e49e0d5307", "docs/source/reference/markup.rst": "7d8effaaf9476135a21d2c4ff94a8e79f63f1049c8a56576bf089ee4c8b74765", "docs/source/reference/measure.rst": "1cc35c3ee641a60e08bdb9dae529cfa29887903ec4689e05323ae1525c893855", "docs/source/reference/padding.rst": "948b448f81a3c22923ad168b28ab13cbfb20a3b1c155c26831d10c18dd640925", "docs/source/reference/panel.rst": "9e76e09001f69fed587f526b879e00cd5eb5ee413169924eee285f656890d6a8", "docs/source/reference/pretty.rst": "4369e64e1ac04cf36cf014318af399ef74fde96de71ecc9526bd60ba9f2fd542", "docs/source/reference/progress.rst": "339080843960943e5cfe8dfbce3f3feb15d7652e36766bc53ac51de8b0bffa70", "docs/source/reference/progress_bar.rst": "93565914240c5784943e7b681df2c35d3476ff195ba695e8a704f46ca13d46ab", "docs/source/reference/prompt.rst": "cf90fe95ecc950781bfc22d78a5fa327e3fbd396f9dcb77833e6aef901ade0dc", "docs/source/reference/protocol.rst": "50171edbdbbd30869886daf1dee1c3ae5494e312b0f70c597c2566041807d2bf", "docs/source/reference/rule.rst": "a15790d55c2a47513464a3353c597af1ab9f8167744b53c04fd6da48768925f4", "docs/source/reference/segment.rst": "c27d64596bf63cb5c7bc20b53fc25f396d0c295d57405a5a16d70e6bdca78188", "docs/source/reference/spinner.rst": "958028b728a58d117ae2adebd65cea44fd26abbd6658618df0bdb1ec779d5832", "docs/source/reference/status.rst": "f6b92d212db0bb62d5d10cf23858dce7fb0b67d1b1ee468de49071828d9be655", "docs/source/reference/style.rst": "2ff78b0591a91cf7ae7b3e86a8b105559e0b2c2cb25a2a4bf8a577b550698e3e", "docs/source/reference/styled.rst": "ad65127d0dc9270c81b60d7cf1fc54f6f61b63f62d92fbd35fda86f652277c43", "docs/source/reference/syntax.rst": "0f80802049c441aa41e0abd171473422e1865e6520f6b281551f57db740e7802", "docs/source/reference/table.rst": "ef611282f1d7ff085148ab6c266292bcb9b33cada57114396f679649f606900a", "docs/source/reference/text.rst": "8185c0308a5108d0325b696d39a29753e4ccd59679bdf8616975affec9d39dc1", "docs/source/reference/theme.rst": "e36100772987b842a75cbc666c2ec6515835e0c0acfce8d3a5b95395065dbe7f", "docs/source/reference/traceback.rst": "40426b8fa4a6bc329c672a5a0f9e411d9b875d9df67e460a1a0de28fe0173e4f", "docs/source/reference/tree.rst": "cdac6b0e658762faddfc7fdb427aeee5e72563b44c89bedfe0156889b3517f65", "docs/source/style.rst": "1e0d81573c767b8e0d9a11275cfed77176fdcbee038bc64e7c28b737600c8218", "docs/source/syntax.rst": "f94406c98635dcfac9056e87d531a4a2be208d88f51ad21a25a4710fc574eeb3", "docs/source/tables.rst": "471d4f879a48f44e96b546498c25d939400731103bb65f6621d8da99b64000d1", "docs/source/text.rst": "40b17f6289539ad5d40fdfb643ccc1165df3a0cea656b61e3b956d86dfa5e359", "docs/source/traceback.rst": "8dd22dfd4eff5d6e8784f6132b77294a1888a402346ce23be97b8d310dc2341c", "docs/source/tree.rst": "e3bd5195add02e8393df1c88022776f310d5b36674b92ea8ef3e750c752ddd97", "examples/README.md": "0335204e3c33be1c4ea692bc0631adaca5b99e79e48e18f9ea0b34ceebbd8786", "examples/attrs.py": "c4d015c624123867a4b97f3e357592c0fe7d7efacbb4c8c440c33209eec37e1d", "examples/bars.py": "8cfc0ff0aed7b05fbdaa6649bb865a27f2f8c7b1540da6cdd1be5322cd5a6259", "examples/columns.py": "88670fdef773c27c71a3a437c5746d33c3cff583e72dcfcd25ff3457b9cebcc3", "examples/cp_progress.py": "29eec0b1a7a6704d08a8c7dcd41a693d54b94868b70b3332f6b94019f91b57f6", "examples/downloader.py": "7a14170f113a2063d9876cc8e126252e493fb32067d605bc59e1d04d6681cd6e", "examples/dynamic_progress.py": "7dfb9795f47bcb52815174af0e7dc6d7c8f55b9a0f0f98b67b90463aca03132c", "examples/exception.py": "ab60cb82bb334edfd501c9d523f12528edd92011d67dfb62b50e59a27dabcf6e", "examples/export.py": "1ba5c61760052f88190b0231f73d6a51842efddff5aab782ffc780f4db88a3de", "examples/file_progress.py": "4092e8512dee9ded5fd399548b3c977adf008bdf87360b4dfb04a50a34f12cfd", "examples/fullscreen.py": "83574552ebc264e8412de447c8ddc154379af3ee8c61e3c948333b2be8727ae4", "examples/group.py": "248ffe1620fcd2bb4dd4ceb27ef0ed6b25c8697025157ec02ce869d42638cf83", "examples/group2.py": "df3129a655b3870955faf3f2d229e19351d7923b19321016a0031dd4d0840516", "examples/highlighter.py": "9a0134e87574e5cbed2b881bb7e4cc3499319c9345db35563fb0b5ce74d50a35", "examples/jobs.py": "8400476d2a92a46b642f8b80adca53a4ba03484600d5634af9c59999db98ddbc", "examples/justify.py": "8c99de81c190593cf140f02a513db46429b58b547fa96eb610d9c6515372cfd3", "examples/justify2.py": "67593b5195ddc3dae55eb051ec1a5f955eab97a51b224948d286c2acfdceaa20", "examples/layout.py": "54bf4488b17e6077d73edb75ce5a9f9d2f598863490733da7d731694d6a94557", "examples/link.py": "513fae345cd6a569aed723f95857c930359c25dacbb7ae6a6cc9ee14bc479b7f", "examples/listdir.py": "c56a06d280915cecb2026cc428960d0da65cc5f8dd6fa54845c66abc2c1f0a7d", "examples/live_progress.py": "4d420c81ae2db4c731ed9889bfaab2000a643a2eed17c545cebc8611e2efcfc5", "examples/log.py": "6ef8b9f6422bb8e3d9ac807d5af0f0cde19e8b9762ab27dc6c4d22263b01503c", "examples/overflow.py": "cf3e33b683185b02e1cdd0f3a4d6a08f104bd6612f3bb8152819db258e40d3ae", "examples/padding.py": "d91713a900200e3773a7b2c425144709a475cdd54c2f600ae9e5b91cc6ada258", "examples/print_calendar.py": "ac1c0bcaf6d3b928ab884257c713e7119b6a4d33b41a6f924107d7de8e24461b", "examples/rainbow.py": "d5795e289d2835655ba6ae61e693e89ac7df8b1268444773998df54bf865c362", "examples/recursive_error.py": "e6dc87fdf4de63fd556488080b343c2596a5136b457fc36ed5248ef6dba04d1b", "examples/repr.py": "97fb41e4f461c0efda9ea4825e18a4294f19dd5cec149aa41a8371d9c2da3aaa", "examples/save_table_svg.py": "40cb26a58f46501599eb525a1e2c57b5e207c6306834771dfc6c1a80856c2700", "examples/screen.py": "c1cad80bd37ab3573619a1ac7fe4744aa465fa0c9ec83918fb9b2e57265d7031", "examples/spinners.py": "e58ae3c415aeec5cf43b093783527f83bb295055ccf57924dfa907cf1f8d8343", "examples/status.py": "8c501ec396987abb9f9544264971b903e407531e9e2c51767ab0a122dacf1c42", "examples/suppress.py": "e441bc40cfce5ed57b835b8a1a36b550ede7acd5c47ba800834248290870f4b9", "examples/table.py": "a97cdceea60fe2e0c56591987cd5419df2f7e0cf84959cc43a18469c11f1ba89", "examples/table_movie.py": "8bc2d32286756f7ea726512fe2600f6d1170fcd037f03da9c8b4ba828502d4ef", "examples/top_lite_simulator.py": "26f04af3eb6cd4c0cf2f1a02603e95ceab6518d00c2c575c030f6b5f74cedd45", "examples/tree.py": "195e2983d760e1c7b19ba058a0addeac36f4a7d75543832cd0afe5ac922fd14a", "faq.yml": "64395e45cbd5cfc07e9a6ec6449b79f38b04a2fc17c4949a8edf1a7899e11ec5", "imgs/columns.png": "5add7e8c4b16ac3e7e227bf17737d845e1aaa63187feb4808e4bc0b05d310e93", "imgs/downloader.gif": "7fcfa5a98fb6cad8cf74d9210de876120e4aa855b945e9d6628575b3d08eded4", "imgs/features.png": "691df8673887414ba0c9bd8801880f894e77d78579dfa3cb7d2b6896f7f72730", "imgs/hello_world.png": "85f7666d2867219887604c7c1cea7e45485a0515fa7be497e6c2e96a9aa15c16", "imgs/inspect.png": "10390ff434da437389d084357e9f3dcf394aa81885726abfbfdfafe4e9715709", "imgs/log.png": "68cf3b05488f8e00e1f2be72b1eaeb6edb34f4d9c24f41615fced3ed3bf71e61", "imgs/logging.png": "8ed54337f14607951cce1e879a31f40d4c0c7cb5c43defa0b549513c4efe0d51", "imgs/logo.svg": "46fbd467b4d19a30ebd9058b4f1da062724aac17b47de19f8c8e532ff7cde0a4", "imgs/markdown.png": "4a75de0654a616e080be7300b7895626b8e9f2350785821a16041c74f98b5ab6", "imgs/print.png": "54173bf385c8360d66e15ca7d210d789cd11a781f1549d326a7f4f5c65f8cb7f", "imgs/progress.gif": "234ed16c499cdd06ec48391b08f86a29e85534bd688c787a6027138ce003e8a9", "imgs/progress.png": "d7b257f5af08358cdd48d118af4f358cb80427623db11acc673bfda826195129", "imgs/repl.png": "e82039326a9a98c4d63cd8821c90f0f4718eb9f3b2603da9989b408e684274b2", "imgs/spinners.gif": "c6c8e73bcec09a66c222f7637bc2da233d94c7066d2a5173ad70022dab8dc26b", "imgs/status.gif": "5912a2f2d8657e709bf5526feba1d08f0cff6dce8b689bb3a2ce8dee16b91a64", "imgs/syntax.png": "9864479c64bfcdf044224187fa9c6597902393160a8b6d2664b8900045a748cb", "imgs/table.png": "454abe34565a5e1e8e601101407be2162cfcd1e78510530f7861736404fb643c", "imgs/table2.png": "57c2142569c2d47fdfefd68f8a3d3c7af7fee7594563a81757b9c9f32964d5f7", "imgs/table_movie.gif": "3fcda5eb275e34b67471c7c0e64742d102fe00bbc7b022bb2444aae39fb277af", "imgs/traceback.png": "e125bdcaae65152ab4f4abd2ef56ce6995031b583af2f175cfd6f450c2541107", "imgs/tree.png": "76242485aec1263b67d9edeaaba65009e38da897b0344001983673e8ae1b2d6d", "imgs/where_there_is_a_will.png": "defd75b64f13e7020f2c60393955d95d2918693660dfa4c26eed34dfbdd1f425", "make.bat": "e7302d938728984369c2a45c22e67016c7472a1d46ebd4fee145261aa4bb7e9d", "poetry.lock": "c57dbba1c5e1dc6c2ea08928380f5f9c1b424ded8254e3a3807c486d22dc54ac", "pyproject.toml": "70299383c928ee83118fbc04e4c2d12e1a151422dff69de72e3c9b334836357f", "questions/README.md": "76a4177081b280c26bd97fde6f0ee8573c35d7f6c0dd93078e053c00019113f0", "questions/emoji_broken.question.md": "2347ac8ffc3cfbbfea31726b36db31b18e7faf6b62122c1d5827a8c684859b2a", "questions/highlighting_unexpected.question.md": "49fa39711023fe63ebb377ef40ecd35c54dedf0723f7062713b3c6e297844ffb", "questions/log_renderables.question.md": "7f6d6ec40633e91af7c36446ebbdb6319551d650ecc0d9f67cd0b82cdb6123a0", "questions/logging_color.md": "e76eb64acf31439a31990bc6160948455cf8b1171d56d3f58ec3d320e372639d", "questions/rich_spinner.question.md": "926db76fb23d22992fb2109c6fe88fcb5ea7e0e184b160d11e53dd135ce11e76", "questions/square_brackets.question.md": "0f39e343108538f62e68f70019b4812ad207ad9fa86c62254e58979c4b238214", "rich/__init__.py": "961d9672820e269e4ce7f95b02c494306bfca2225eba644e6b31c7fc060c4bc2", "rich/__main__.py": "9590b9571d1d47b244e4150a0b18ede4f358ba1c57f5aba82e2a3b7f804390aa", "rich/_cell_widths.py": "da7e048898b75fdb2a22ad0ed7a91467fcf2e9460c777c457c286529f9d6d477", "rich/_emoji_codes.py": "86ed552fd9db55da6926b5688a356c85195c4517bfbf7763bb7326776b0a65d6", "rich/_emoji_replace.py": "9fe91c7adb04531d99526850adf78c35cfad79e1a1a6e490e45f153c1b32bc3a", "rich/_export_format.py": "b5ccb77613f90b143fb646d06103794607c8ce920d22eadb781c853fb3ce65e2", "rich/_extension.py": "1bae8f91b1ff41d4c987a8c3f89db6f0eefa0a6027af684b42fef60a03cfbb31", "rich/_fileno.py": "1d66713f90b66a331b1ebcaf01066c79f9557d0a06cec28e1f3286b0b0fcca74", "rich/_inspect.py": "a19246c37d5eeb87705d20a6ac39ef65bc156f564a8567d4f30237556a218c99", "rich/_log_render.py": "c41282c6a88ee05664f1e1b9e9ff1cac576b989c45ac9b1013757717e7c57a47", "rich/_loop.py": "855ffa08b7683e6d2f6b6d96a70e332aa334458b33dd36715e3d0fa12fbd7834", "rich/_null_file.py": "b4649793fbfe21999b8f5180cc78adf00de460840c882a55b0215fb02fbf289e", "rich/_palettes.py": "71d7afd4940a67426f960b95f62a478339d3767be52335050c16f422dd8fce32", "rich/_pick.py": "7af0edf10378945e428b0ad421794e2429ed8ad0423ac23764b3c42005512c95", "rich/_ratio.py": "e9d047c62c3f49835145d719f600ca2391a701e1d9f16be64e072d36972eb665", "rich/_spinners.py": "536af5fe0ff5cd28ec8e251d00449cda200c7378b8ae2fd2f0f60fea4439cf52", "rich/_stack.py": "f82f0e2bbaf19f7b0851d570c59041a5e1e12335f4788f9533731e9987da5e6d", "rich/_timer.py": "cde9716d3ea83c566736bc163e973592d51e013f957387ee15c4592d018bb4c2", "rich/_win32_console.py": "f8db5e6a5b105d0e79f1efb3a49b64c21f31fdda82f928e500603cef674613a5", "rich/_windows.py": "27fae1687ec73377f95ef1f96ec02b18aaa84e88da57c1deb4d2d4275497f48e", "rich/_windows_renderer.py": "77bf7dc4e9f12db0820b31aef7e53b60b988424c6dc5020115043a8aee2f7927", "rich/_wrap.py": "c5f57ff6dd1283aaf38a69ab0ebbbc7c25665256a56007072c37eb2599db6f04", "rich/abc.py": "7402cc3867ca54d7806efaaaeba2294d0c5051eaf10fb004e052b0a9dd1e40a9", "rich/align.py": "e73e47bd4dfb268468e7bf59d92b5d8f711ac2cd5c418a266ea401fad4f75636", "rich/ansi.py": "883eb9df6418aa7066ea1003ba52a3ad5f25f24149fbd7c4568a072471f784c8", "rich/bar.py": "6bb503df4dc171c442ac48468df304969bf94456088a7680840baa62a854be6c", "rich/box.py": "b0589cbd122159119d44ce5adfa5df7c3899ba4b12f4857cef077f14e6d7590d", "rich/cells.py": "eb6ef3b49b3dcce2fedfc1c9ee45c17ab47e813f0a05f602f14cc4c0c243618a", "rich/color.py": "f4687de7c5377fbe5655d2d3782d14f679064e7da7d309e88caa49ea34049081", "rich/color_triplet.py": "de585091d25bbd63e82c33be0276089805a626f579765818342559f7b39168de", "rich/columns.py": "1d45f429c326f5db0a362d757d36e233f876883b65f3248269573195a944ceaf", "rich/console.py": "123176a83e50377257830fa32e16d00715e8a1fe223077f3417f37e9ccb931c0", "rich/constrain.py": "d5520fb82f0082d296adc9dc42b8c1758a80dc9556cacbba8d9a35aeb87b73b4", "rich/containers.py": "68a826e540c79f9366ba2e8825a29db1985b1c2961fd7ec3fbf5a0f0486bafbb", "rich/control.py": "231fab3bc661481daad4189acebe25ef6672030dbb1fda2bec4962a56555a343", "rich/default_styles.py": "881134e1b0fa449095865a5491e0a7127ba59b2759ccaa950311ccf86a685c12", "rich/diagnose.py": "668a43d84a5656d9a62a9b606d74feb0e324009ec31ab312517522694d8667bf", "rich/emoji.py": "d634d11c5c2f431635722ba5db631d11971673b6eb7e3293f051ba6635e3e5d3", "rich/errors.py": "e693f729ce5de1027f734285b31adfca18e23d57bb275ccea9215b140cdc57e6", "rich/file_proxy.py": "4e5f531cc0d9f8f9395a6f2c23580683f5390e1bac9b10fe159d1f51b714d16d", "rich/filesize.py": "f5f4cb00f080c079815dd46feca654d7de234a036b45be96c7b448a0182a78a6", "rich/highlighter.py": "a770b5838418cdecc529d47b345f4484f6f3403bdd3d48464604b21861263e4a", "rich/json.py": "b5c4195b250d16e8f1628374880aa36ffc2d205f9e52176e6973a142bf82689c", "rich/jupyter.py": "1bda4e26647811220561277830a1aa987a82b6cb71d28456a7279382fe78f977", "rich/layout.py": "f9a6d2f2bbe3aa57ae86fd98130c27a2de157fbc835ecd1b8d4e78ae8b448642", "rich/live.py": "bd9cd8beeedfab096fdc6b61976c62c350dcfcef3456519c095d03387c02c833", "rich/live_render.py": "bd27fbb2eff0ad7cd96063310210803e81898024e39a5c5c53d1b8bd28a3b9c5", "rich/logging.py": "5c4294b5e09984dee3d299889e04ce081498d86bee4f8ff8f85db4fa26cfac49", "rich/markdown.py": "5765e65143b3602ac390946d591b62f66360d2fdb8020942baeeccf076e8887c", "rich/markup.py": "79f8156256cd61fe123f8ce08afddef116e1c6bd50193b34962adc1f0ba16475", "rich/measure.py": "1e6ac8257f2c5914c76e087c33111acbff37564a8d5bfef4b3c68a3f965c608f", "rich/padding.py": "f08d5d40d4102b58828c7b46043127efc911a77a76ec6633979511dea07b7b61", "rich/pager.py": "48efc44c114a6e0de7fc080ecd79b8d52bf7e98c57032237fd1f8a398dbfb927", "rich/palette.py": "02be9952b607885b7af91af693e93d17c57b87180960735daa3936bd55ec2e07", "rich/panel.py": "c0631ee3427c2821a04283342f28d112b986224bf66ec600ef54425d3843d311", "rich/pretty.py": "e5e7abcdd21fc4a67bb4ba195bc08d6f75ab75a5e4a89b0e6a45ac4483b8dc40", "rich/progress.py": "9d68e8f6660ca93fe66c713a42fdb24e192db94b22b110707026e250139d8659", "rich/progress_bar.py": "704a017e473794bc2a6dae172ac529cb8bd240a0e1d9043927627de3e002168a", "rich/prompt.py": "53ded4063097059ed39884692be27190893d8abbc9888bbf1c1da967651a43dc", "rich/protocol.py": "5adfb61d977aece622a295240933b3ee5337f2fca8e6bdc711067d4ce3c39794", "rich/py.typed": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "rich/region.py": "acd4fdc59ad56536085d90b43589f8d42250c1835b47e29e70f3b14e042f07c6", "rich/repr.py": "1c8b2eacf2d92bd1ab6b2ef9977fef431ed2dbb0334e9023e0285749f7b515eb", "rich/rule.py": "ba63b6d568f0d0571801e4c1dd4ba634b0ac0d685e8f3c678e57f65708978832", "rich/scope.py": "95fe907adfdee09398df89708584801be3415d8281718bbab7f8a0bff2681a88", "rich/screen.py": "acbfe3db05fee127ae20e236a0e95ce35f103fd100bc3e7d1889d4984004ea34", "rich/segment.py": "7fb537a5571ced3a57b30d0b987ead7016e154805198d45c0db42f957d331900", "rich/spinner.py": "d799280a61740d0783f3e936f0ba6de97ff3250525cc4860a3fe80eaecb8ee57", "rich/status.py": "809b085c865e4a8deeacecb14548ece95ae15f9099ac0d0dc4843e7718429f0a", "rich/style.py": "de18a8707ff837cbf0466dfef32156ccceed4b08e312f7a7ebd5ea59ab124303", "rich/styled.py": "c258d5b154d76c004c319be4ce43b8dd9124ffe8abcc4b6f52243eb0d9e2910f", "rich/syntax.py": "9a38be6e37e9f8a7caf77ca1bc6f15efb6b533d6bcef6fef750f1f7a2c967ce5", "rich/table.py": "4eec6e954200c1bc25bd23c06f525866db8c8cb92b75296c35382fceab6a76cc", "rich/terminal_theme.py": "d63e7eb9f25f9ef940a3942c8bf0026625c39b0317cea826141c8e6d3f7ec896", "rich/text.py": "6234da5c6a12ef069c66a5fd905d87e3ace7274daac78eea7901b4f52a8b9f2a", "rich/theme.py": "6de9452688330345b41f2b1069b29a1ce7374561f6928ddf400261a0df8015da", "rich/themes.py": "d318132e8cdf69b79b62d709b43742e50917e4855411abe2a83509261e185459", "rich/traceback.py": "b4124fe2ca292386eb42d94cad2f94cd9c580b3a1478294b7cb26f7ece0dd452", "rich/tree.py": "02cdc19225fb571e054244ce167b6b7fa94c2404d95cad02f9ced1bb12fa2f21", "setup.py": "1544f95379a1abac196f3abb11c613e3fba76259697a2c4d33b47246e3b3f91c", "tests/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "tests/_card_render.py": "856eed79723d6687f221b684c2a55e2fcd37ce2a4287231f30e36e74681b5630", "tests/_exception_render.py": "94ce6a4e31912b8209d88f73e743cbf79f957d0f0de5c3f8ba962a14a4d0f1ba", "tests/pytest.ini": "54f8c40ca737dd81c7318088cee5d6465fbe5e727ea3281dccac1028073ddb5f", "tests/render.py": "fcead9339461fba999c428808db5efc4cc8a0dddabab0b45f6e6d1b0f4a4e14f", "tests/test_align.py": "8ea0fcac298b5d96a8c58c7f9b8392024b79006a132abcc8cbb615ff53521c28", "tests/test_ansi.py": "77aff75ce9681b2f5c8fb345f6df3db03d27988c11435094bbca203fa5d5ab97", "tests/test_bar.py": "1a17cfdd1892a46f78b9c31162cef1d1208a5f3466bb1ec79ea4216692ba59d3", "tests/test_block_bar.py": "ca07f952133d474f9311fbaa6baeea6884d4fd93fd69035efb8d462f8e39a468", "tests/test_box.py": "aca1ced46876eb822cbfce84521ad3f2e0229c4a51a504c84592f65ee8c70cc9", "tests/test_card.py": "a53a255bdc033ce56eccd17d76c08e0aad4dc3bc1605f3b0b02f9e6c520173f0", "tests/test_cells.py": "d9b7aab6763c9f75f1de71b6558c434641ffadba1a4f3e8ce3b1c7997c8157dc", "tests/test_color.py": "8e9a76d9dae7c8850e2dfe667ad41be5273e67c9e124244df120c4607dbc5df7", "tests/test_color_triplet.py": "251450b92cb09288cee6c72bed534fcb373843b9df96423a1430c4b3ec4ac77c", "tests/test_columns.py": "62dfd317911602359aea0d6efba8a45e409062dcc9bb2c634800c062ce10e67c", "tests/test_columns_align.py": "46fbc972781e1ba8d0e2da12fbae122667c90039817214d1193fdc1bcf42b9a2", "tests/test_console.py": "1a37ecdc10aded197a422d2d027ae20e3225804817408c0de3c7bd1e3ccecf5d", "tests/test_constrain.py": "670a716bd5a22927ecc7ae3ee143300ad958027015e7b264350effd30ba118c7", "tests/test_containers.py": "dbb5a22c118fe3cd19b22cc4668fe3cb342b813fa2ca86e28cf8b568ae4aa141", "tests/test_control.py": "a25bf2636cdd1c59464a0e385a695332134b433ac2957d50f580a41ef99ec647", "tests/test_emoji.py": "ae5d5d4ee3a77749661bdd1936f9a7f315752a882dcb5af419627981516ca881", "tests/test_file_proxy.py": "96c50256ebe7a6e4b166fe7efe87c1a55c5450ebbb8e4cb89bb15e9ec4ac26b6", "tests/test_filesize.py": "01e6636da4c9626be8b33cfabbf6ee0531df1e5515ca418ecd66ad5bd85e8a65", "tests/test_getfileno.py": "093be26c44abd319adfc00686618a92dad0550f749ce826767200885b1753145", "tests/test_highlighter.py": "c96124bdc4db430017b956bd51dcf593e87db98c17b77d2c16868d6e5f188c71", "tests/test_inspect.py": "f5e7870c881a8052479494fdefb5ec7089772ea196242b97877060d8523bd1da", "tests/test_json.py": "06deeb613baae897f86fc2e1ee6e8bf547cc4e35cefb2bdb8e2fbc773bc7aa2d", "tests/test_jupyter.py": "ee6289bae3dd61f1d0e26daef0f029abb11ddea5c6982747f8381748bcb76efa", "tests/test_layout.py": "3ad988c9ec309cdb84400b394a3b3862e1c2037c8537bfd2eedb069c843db736", "tests/test_live.py": "1291d88fa847ce70309e1c50b5e558aa5b29903faf7e99e41a25b4f74e4fcd81", "tests/test_live_render.py": "37281f149f4df496cc67324cab39b2bb68c0d4d02a6d3d34a2fb7fcd2c1c4a14", "tests/test_log.py": "d949111aced8ad28b35ae4fceed73278a0832b8f3e6de3746eeeb2cb68e4b297", "tests/test_logging.py": "2cb3bd8c5468377f3ce379ef31140a2efacf0a8643a502d33ce6a214fd3e85fd", "tests/test_markdown.py": "2170f192ac63b50d4acba0483cd9867016e24bd4f6910de04d65f71f78cffbb1", "tests/test_markdown_no_hyperlinks.py": "0c252b455def34ce978be475d4687b7b3f9f93209979fae266f6ab5f21d287cd", "tests/test_markup.py": "6d4211e5c673ea2b72622ef9e4e7fb064f6373a0aaa811c7c6dcfdf5257f5d68", "tests/test_measure.py": "aec9e557b48eb3c69c718d378e59667966ae5861e639796affec6e0dd6f4083e", "tests/test_null_file.py": "e13cfd1caf61f8eb031432a572e927ec6c283d09e10cc2d640d62f102b372c29", "tests/test_padding.py": "0f8604b9078963f45cf9e8dcd5d76c72b58d5fbe9a1f9630f58b41daf03a0781", "tests/test_palette.py": "869b240032f18de48a73dc434781acbb2fb84f1184f97ab5a8b063f71053ab61", "tests/test_panel.py": "572c36143e8fa0c0d61a716819aedb515cda7d48d0931133f80418e770704904", "tests/test_pick.py": "a74a055a506cb6af9e11d3035feb29eb9e987e1edc1b325617f51b64767c36a3", "tests/test_pretty.py": "cba41cdee406c2d51871fb2527f2cdd3e9b372b240c530e1ffb6841539d10ce1", "tests/test_progress.py": "1c0fc0013d4ee8def1b08ba840efb52037daecc10665c8c1a1104e223405e08c", "tests/test_prompt.py": "35ee2c8d8c91d5cface3fccbbff1fa68708a7a3ccbb999a70029587d37485b05", "tests/test_protocol.py": "6645dfa6e722bb585679839dfd0250f7990e9e81bf0e4d307b54e7acdef1463b", "tests/test_ratio.py": "e4cd4026c0a6659626beb2939f085fc0a787eedbdc0a596da641e555b95afafd", "tests/test_repr.py": "f0b9ca28cf30b40d845e9de8a511e57555689e6979786dd4ce42459f3a29cd68", "tests/test_rich_print.py": "90f13cf555d7abe3c69fbcc6808090fba567bff1dbb4cd0091dde9b52c4c324f", "tests/test_rule.py": "adf08860b6b18dfc3f317e2ffed7e0db629819ac8b1f77698a2c3c98adac641b", "tests/test_rule_in_table.py": "20db50b992643a5e4f918b0e8f3a0822d338fc89268d63197ca4513b086559fc", "tests/test_screen.py": "ea25b900027dba18ed9baf4c2f6be59715eafc7bfe67dd273ecb18ed9f6e410e", "tests/test_segment.py": "d21b318d227eb351bb85396ac3807fc1cc837f2a0d753020b35a96d0228639b7", "tests/test_spinner.py": "952e980b0e0dff009ccfd8dc08025bbc6bd20cd12ac8ec5032b494e2142fe81e", "tests/test_stack.py": "fb289a55d7c2b56bb5f996fd6930cf0066b1f54c1222d75d560e89cdd07d9ed0", "tests/test_status.py": "a673c36fe9d52fa077f0d07ad49d03774167be90e132173bcc7b7814a2263cc4", "tests/test_style.py": "87e50610d4b808d9f82d7be797cfe6cc849a345f2e1ae5b07b17235dfbeb6aed", "tests/test_styled.py": "05c47db0a385bfe12283670295e1c65ee054ac83b038dbb79451f700be9f420d", "tests/test_syntax.py": "7483ed5de7891c41cc9416c96aed7feb3b75536fc5aff859e232b6f569ca99ef", "tests/test_table.py": "303586b97657fa838ef2124ad926a1ec9f4725ba6f98da353a7dfa325982edcd", "tests/test_text.py": "adf131d5df0f67678bc021d0df581a4a00d190c20eb25de2cf3675e4f03f9e17", "tests/test_theme.py": "67b02502bd4a3b68b7df18199756a107f981e152637a89cfe2ac5e610d70f03f", "tests/test_tools.py": "54ca44adcedc1e50d66d63ad6a8335a82192acd50ca09dde9d63851d72b4eba7", "tests/test_traceback.py": "dd66b21350c39ace75bebafe07722cbbf0318004524b4d70a8288a4f4a7710e0", "tests/test_tree.py": "fc180eb36f20a7252cc3997528ecf6fc6949fb52c4e69b7577eba04aae0c7077", "tests/test_win32_console.py": "270c22e211126524ac3f3809e1a20205c252ed7923df9c04db91b8b18331a483", "tests/test_windows_renderer.py": "85283e90d344b410aec40086aa35d29f728d23882efae7ba37bc41d4d153aebc", "tools/README.md": "dbd7fd223804e511b3c6b6bc142ce70faa7f4a8110d66789047cae9fd5bd7f88", "tools/cats.json": "4f98c2c20a4b44d763681b3c3cea30a1f427dccb85ed0d8f23e7721f97872e6d", "tools/make_emoji.py": "d8383bc47950e3e705a155d865b87ccc27a2b3e88ab19b6da129fd456d95b01b", "tools/make_terminal_widths.py": "6485e6f9e7f4cb34daa76289c8b5ac8b6cf0765565cd97f2ff01e5bde2aa8536", "tools/movies.md": "603d60ee1f25a012ffcaed36f526f3a3f1c9326d1a0519027c126dd844ffd057", "tools/profile_divide.py": "abe4246d9a0486b769bd35f27a10c1b814c841524e3d98ba225a6786d1b1754f", "tools/profile_pretty.py": "0b33dccf00feb6686e99213ee63d995297d06b3f421e1d68b30b830572e1e620", "tools/stress_test_pretty.py": "0e30dc5d5044c4097e290fda96981a1ae1ccc8dbf6f8d2ee89a504e09d2a1337", "tox.ini": "62722eda5de44fbf03c9c3d6678a5d4aa08604db0809403f683cbed42cbc21a4"}, "source_bytes": 19416123, "reference_fields_included": false, "independent_task_validation": null}, "problem_only_task": {"base_commit": "1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d", "hints_text": "", "instance_id": "rich_3067", "problem_statement": "Allow tilde in URL (fixes #3057)\\n\\nThis PR fixes #3057.\\r\\n\\r\\n## Type of changes\\r\\n\\r\\n- [X] Bug fix\\r\\n- [ ] New feature\\r\\n- [ ] Documentation / docstrings\\r\\n- [x] Tests\\r\\n- [ ] Other\\r\\n\\r\\n## Checklist\\r\\n\\r\\n- [x] I\'ve run the latest [black](https://github.com/psf/black) with default args on new code.\\r\\n- [X] I\'ve updated CHANGELOG.md and CONTRIBUTORS.md where appropriate.\\r\\n- [x] I\'ve added tests for new code.\\r\\n- [X] I accept that @willmcgugan may be pedantic in the code review.\\r\\n\\r\\n## Description\\r\\n\\r\\nPlease describe your changes here. If this fixes a bug, please link to the issue, if possible.\\r\\n", "repo": "Textualize/rich"}, "mobile_input_manifest_sha256": "5b0a706e1e77381c0486933490afae37eb3562b850256b5154e3e05a18bc6447", "input_packet_sha256": "19ae6523b9b420443f83871fcc1329d5770ba79d42c8d2b6fa8a14edce57cb29"}')


In [ ]:
if IS_KAGGLE and RUN_SOURCE_AUDIT:
    RUN = Path('/kaggle/working') / ('native_source_audit_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    report = run_mobile_source_audit(DATA_DIR, RUN, PLAN, export_run)
    print('MOBILE_SOURCE_AUDIT_RESULT=' + json.dumps(report))
else:
    print('Definitions only; no official source audit executed')
